유저 세팅1

In [ ]:
from pypylon import pylon

def inspect_user_set():
    # 1. 카메라 연결
    try:
        tl_factory = pylon.TlFactory.GetInstance()
        camera = pylon.InstantCamera(tl_factory.CreateFirstDevice())
        camera.Open()
        print(f"📷 연결된 카메라: {camera.GetDeviceInfo().GetModelName()}")
    except Exception as e:
        print(f"❌ 카메라 연결 실패: {e}")
        return

    # 2. User Set 1 불러오기 (Active Layer로 로드)
    print("\n🔄 [User Set 1] 설정을 불러옵니다...")
    try:
        camera.UserSetSelector.SetValue("UserSet1")
        camera.UserSetLoad.Execute()
        print("✅ 로드 완료. 현재 카메라에 적용된 값을 읽어옵니다.\n")
    except Exception as e:
        print(f"❌ User Set 1 로드 실패: {e}")
        camera.Close()
        return

    # 3. 주요 설정값 읽기 및 출력
    print("="*40)
    print("   User Set 1 저장 내용 상세 리포트")
    print("="*40)

    # (1) 노출 시간 (가장 중요)
    exposure_time = camera.ExposureTime.GetValue()
    print(f"1. 노출 시간 (Exposure Time): {exposure_time:.1f} µs ({exposure_time/1000:.1f} ms)")
    if exposure_time > 16666:
        print("   ⚠️ 경고: 노출 시간이 16.6ms보다 깁니다. 물리적으로 60 FPS 불가능!")
    else:
        print("   ✅ 상태 양호: 60 FPS 가능한 짧은 노출 시간입니다.")

    # (2) 프레임 레이트 (FPS)
    target_fps = camera.AcquisitionFrameRate.GetValue() if camera.AcquisitionFrameRateEnable.GetValue() else "Off"
    resulting_fps = camera.ResultingFrameRate.GetValue()
    print(f"2. 설정된 목표 FPS: {target_fps}")
    print(f"3. 실제 가능한 FPS (Resulting): {resulting_fps:.2f} fps")

    # (3) 이미지 크기 (ROI)
    w = camera.Width.GetValue()
    h = camera.Height.GetValue()
    off_x = camera.OffsetX.GetValue()
    off_y = camera.OffsetY.GetValue()
    print(f"4. 해상도: {w} x {h}")
    print(f"5. 오프셋: ({off_x}, {off_y})")

    # (4) 픽셀 포맷
    pixel_format = camera.PixelFormat.GetValue()
    print(f"6. 픽셀 포맷: {pixel_format}")

    print("="*40)

    camera.Close()

if __name__ == "__main__":
    inspect_user_set()

폴더 리스트 추출

In [ ]:
# 파일명: find_relative_folders.py
# 지시사항: 지정한 위치 바로 아래에 있는 폴더들만 찾아서, 현재 실행 위치(CWD)로부터의 상대 경로를 'TARGET_DIR = r"상대경로"' 형식으로 출력합니다.

import os

def print_relative_directories(search_path):
    # 현재 실행 중인 위치 (Current Working Directory)
    current_dir = os.getcwd()

    # os.scandir을 사용하여 지정한 경로 바로 아래에 있는 항목만 탐색 (재귀 탐색 안 함)
    for entry in os.scandir(search_path):
        if entry.is_dir(): # 해당 항목이 폴더인 경우에만 처리
            # 현재 위치(current_dir)를 기준으로 폴더의 상대 경로 계산
            relative_path = os.path.relpath(entry.path, current_dir)
            
            # 요청하신 형식에 맞춰 출력
            print(f'TARGET_DIR = r"{relative_path}"')

if __name__ == "__main__":
    # 탐색할 폴더 경로를 지정하세요. 
    SEARCH_LOCATION = r"data\basler_0409" 
    
    if os.path.exists(SEARCH_LOCATION):
        print_relative_directories(SEARCH_LOCATION)
    else:
        print(f"지정한 위치를 찾을 수 없습니다: {SEARCH_LOCATION}")

검증 

In [65]:
# 파일명: verify_all_recordings_with_table.py
# 지시사항: 하위 폴더들을 탐색하여 검증을 수행하고, 맨 위에 전체 요약 결과를 표(Table) 형태로 출력한 후 상세 로그를 보여줍니다.

import os
import glob
import pandas as pd
import numpy as np
import json 

# =========================================================
# [설정] 검증할 상위 폴더 경로
# =========================================================
PARENT_DIR = r"data\basler_0409"

TARGET_SENSOR_HZ = 200  # 센서 하드웨어 전송 속도는 고정 (200Hz)
# =========================================================

def load_settings_from_txt(target_dir, log_func):
    """설정 파일을 읽고 결과를 출력 대신 log_func(버퍼링)으로 넘깁니다."""
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_duration = 60.0
    default_fps = 60.0
    default_version = "Unknown"
    
    if not os.path.exists(settings_path):
        log_func(f"⚠️ 'camera_all_settings.txt'가 없습니다. 기본값(60초, 60fps)으로 검증합니다.")
        return default_duration, default_fps, default_version

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            
            duration = summary.get("Record_Duration_sec", default_duration)
            fps = summary.get("FPS_Result", summary.get("FPS_Target", default_fps))
            version = summary.get("Program_Version", default_version)
            
            log_func(f"📄 [설정 파일 로드 성공]")
            log_func(f"   - 수집 환경: {version}")
            log_func(f"   - 녹화 시간: {duration} 초")
            log_func(f"   - 설정 FPS : {fps} fps")
            
            return float(duration), float(fps), str(version)
            
    except Exception as e:
        log_func(f"⚠️ 설정 파일 파싱 실패 ({e}). 기본값으로 진행합니다.")
        
    return default_duration, default_fps, default_version


def process_folder(target_dir):
    """단일 폴더를 검증하고, [요약 딕셔너리, 상세 로그 문자열]을 반환합니다."""
    logs = []
    def log(text):
        logs.append(text) # 화면에 바로 인쇄하지 않고 모아둡니다.
        
    folder_name = os.path.basename(target_dir)
    
    # 요약 표에 들어갈 데이터 구조
    summary = {
        "Folder": folder_name,
        "Mode": "Unknown",
        "Cam_Target": 0,
        "Cam_Actual": 0,
        "Cam_Result": "❌",
        "Sen_Target": "-",
        "Sen_Actual": "-",
        "Drop": "-",
        "Sen_Result": "-"
    }
    
    log(f"\n=================================================================")
    log(f"🔍 데이터 검증 상세: {folder_name}")
    log(f"=================================================================")
    
    if not os.path.exists(target_dir):
        log(f"❌ 폴더를 찾을 수 없습니다: {target_dir}")
        return summary, "\n".join(logs)

    # 1. 파라미터 로드
    actual_duration, target_fps, program_version = load_settings_from_txt(target_dir, log)
    summary["Mode"] = program_version
    
    expected_cam_frames = int(actual_duration * target_fps)
    expected_sensor_data = int(actual_duration * TARGET_SENSOR_HZ)
    summary["Cam_Target"] = expected_cam_frames
    log("-" * 40)

    # -----------------------------------------------------
    # 2. 이미지 파일(.tiff) 검사
    # -----------------------------------------------------
    frames_dir = os.path.join(target_dir, "frames")
    if not os.path.exists(frames_dir):
        log("❌ 'frames' 폴더가 없습니다! (혹시 .raw 저장 버전인가요?)")
        tiff_count = 0
    else:
        tiff_files = glob.glob(os.path.join(frames_dir, "*.tiff"))
        tiff_count = len(tiff_files)
    
    summary["Cam_Actual"] = tiff_count
    log(f"📸 [이미지 파일 검사]")
    log(f"   - 파일 개수: {tiff_count} 장")
    
    margin_cam = max(10, int(expected_cam_frames * 0.03)) 
    if abs(tiff_count - expected_cam_frames) <= margin_cam:
        log(f"   - ✅ 판정: {target_fps}fps 카메라 저장 양호 (목표: {expected_cam_frames}장)")
        summary["Cam_Result"] = "✅"
    else:
        log(f"   - ⚠️ 판정: 카메라 프레임 수 부족/초과 (목표: {expected_cam_frames}장)")
        summary["Cam_Result"] = "⚠️"
    log("-" * 40)

    # -----------------------------------------------------
    # 3. 카메라 타임스탬프 검사
    # -----------------------------------------------------
    cam_csv_path = os.path.join(target_dir, "camera_timestamps.csv")
    if os.path.exists(cam_csv_path):
        df_cam = pd.read_csv(cam_csv_path)
        cam_start = df_cam['Timestamp'].iloc[0]
        cam_end = df_cam['Timestamp'].iloc[-1]
        cam_duration = cam_end - cam_start
        cam_actual_fps = len(df_cam) / cam_duration if cam_duration > 0 else 0
        
        log(f"🎥 [카메라 로그 분석]")
        log(f"   - 기록된 프레임 수 : {len(df_cam)} 개")
        log(f"   - 로그상 녹화 시간 : {cam_duration:.2f} 초")
        log(f"   - 계산된 평균 FPS  : {cam_actual_fps:.2f} fps")
        
        if tiff_count == len(df_cam) or tiff_count == 0: 
            log("   - ✅ 파일 수 == 로그 수 일치 (또는 단일 .raw 포맷)")
        else:
            log(f"   - ❌ 불일치! (이미지: {tiff_count}장 vs 로그: {len(df_cam)}줄)")
            summary["Cam_Result"] = "❌" # 불일치 시 실패 처리
    else:
        log("❌ 'camera_timestamps.csv' 파일 없음")
        summary["Cam_Result"] = "❌"
    log("-" * 40)

    # -----------------------------------------------------
    # 4. 센서 데이터 분석
    # -----------------------------------------------------
    if program_version == "CameraOnly":
        log(f"⏭️ [센서 데이터 분석 생략] 'Program_Version'이 'CameraOnly'입니다.")
        summary["Sen_Result"] = "⏭️ (Skip)"
    else:
        sensor_csv_path = os.path.join(target_dir, "ppg_sensor.csv")
        summary["Sen_Target"] = expected_sensor_data
        
        if os.path.exists(sensor_csv_path):
            df_sensor = pd.read_csv(sensor_csv_path)
            sen_start = df_sensor['Timestamp'].iloc[0]
            sen_end = df_sensor['Timestamp'].iloc[-1]
            sen_duration = sen_end - sen_start
            sen_hz = len(df_sensor) / sen_duration if sen_duration > 0 else 0
            
            summary["Sen_Actual"] = len(df_sensor)
            
            log(f"📡 [센서 데이터 분석]")
            log(f"   - 데이터 개수    : {len(df_sensor)} 개")
            log(f"   - 로그상 녹화 시간 : {sen_duration:.2f} 초")
            log(f"   - 평균 샘플링    : {sen_hz:.2f} Hz (목표: {TARGET_SENSOR_HZ}Hz)")
            
            margin_sensor = max(50, int(expected_sensor_data * 0.04)) 
            if abs(len(df_sensor) - expected_sensor_data) <= margin_sensor:
                log(f"   - ✅ 데이터 양 양호 (목표 {expected_sensor_data}개 기준)")
                summary["Sen_Result"] = "✅"
            else:
                log(f"   - ⚠️ 데이터 양 이상 (목표: {expected_sensor_data}, 실제: {len(df_sensor)})")
                summary["Sen_Result"] = "⚠️"

            if 'Frame_Index' in df_sensor.columns:
                seqs = df_sensor['Frame_Index'].values
                diffs = (np.diff(seqs) - 1) % 256 
                total_drops = np.sum(diffs)
                summary["Drop"] = total_drops
                
                if total_drops == 0:
                    log("   - 🏆 [Perfect] 패킷 드랍 0개! (센서 무결성 검증 완료)")
                else:
                    log(f"   - 🩸 [Warning] 총 {total_drops} 개의 패킷이 유실되었습니다.")
                    summary["Sen_Result"] = "⚠️"
            else:
                log("   - (Frame_Index 컬럼이 없어 정밀 드랍 체크 건너뜀)")
        else:
            log("❌ 'ppg_sensor.csv' 파일 없음")
            summary["Sen_Result"] = "❌"
            
    log("\n✨ 해당 폴더 검증 완료")
    
    return summary, "\n".join(logs)


def verify_all_folders():
    if not os.path.exists(PARENT_DIR):
        print(f"❌ 상위 폴더를 찾을 수 없습니다: {PARENT_DIR}")
        return

    subfolders = [f.path for f in os.scandir(PARENT_DIR) if f.is_dir()]
    
    if not subfolders:
        print(f"⚠️ '{PARENT_DIR}' 내에 하위 폴더가 존재하지 않습니다.")
        return

    print(f"🚀 전체 폴더({len(subfolders)}개)의 검증을 스캔 중입니다. 잠시만 기다려주세요...\n")
    
    summaries = []
    detailed_logs = []
    
    # 모든 하위 폴더를 순회하며 백그라운드에서 요약 데이터 및 로그를 수집합니다.
    for folder in sorted(subfolders):
        summary_data, log_text = process_folder(folder)
        summaries.append(summary_data)
        detailed_logs.append(log_text)

    # -----------------------------------------------------
    # 💡 [출력 1] 요약 테이블 (맨 위)
    # -----------------------------------------------------
    df_summary = pd.DataFrame(summaries)
    
    print("=" * 95)
    print("📊 [전체 데이터 검증 요약 리포트]")
    print("=" * 95)
    # DataFrame을 표 형태로 정렬하여 예쁘게 출력합니다.
    print(df_summary.to_string(index=False, justify='center'))
    print("=" * 95)
    print(f"👉 총 {len(summaries)}개 폴더 검증 완료\n")

    # -----------------------------------------------------
    # 💡 [출력 2] 개별 상세 로그 (테이블 아래)
    # -----------------------------------------------------
    print("\n\n" + "=" * 95)
    print("🔎 [폴더별 상세 검증 로그]")
    print("=" * 95)
    for log_text in detailed_logs:
        print(log_text)


if __name__ == "__main__":
    verify_all_folders()

🚀 전체 폴더(19개)의 검증을 스캔 중입니다. 잠시만 기다려주세요...

📊 [전체 데이터 검증 요약 리포트]
     Folder        Mode     Cam_Target  Cam_Actual Cam_Result Sen_Target Sen_Actual Drop Sen_Result
20260403_153112 CameraOnly     1199        1201         ✅           -          -     -   ⏭️ (Skip) 
20260403_153547 CameraOnly     1199        1200         ✅           -          -     -   ⏭️ (Skip) 
20260403_153712 CameraOnly      800         800         ✅           -          -     -   ⏭️ (Skip) 
20260403_154017 CameraOnly     1199        1200         ✅           -          -     -   ⏭️ (Skip) 
20260403_154225 CameraOnly     1200        1200         ✅           -          -     -   ⏭️ (Skip) 
20260403_154512 CameraOnly     1199        1200         ✅           -          -     -   ⏭️ (Skip) 
20260403_154645 CameraOnly      800         800         ✅           -          -     -   ⏭️ (Skip) 
20260403_155010 CameraOnly     1200        1200         ✅           -          -     -   ⏭️ (Skip) 
20260403_155121 CameraOnly     1199  

시각화  - 사진, PPG

In [ ]:
import cv2
import pandas as pd
import numpy as np
import tifffile
import os
import json # 💡 JSON 파싱을 위해 추가

# =========================================================
# [설정] 재생할 폴더 및 옵션
# =========================================================
TARGET_DIR = r"data\basler_0409\20260403_153112"
TARGET_DIR = r"data\basler_0409\20260403_153547"
TARGET_DIR = r"data\basler_0409\20260403_153712"
TARGET_DIR = r"data\basler_0409\20260403_154017"
TARGET_DIR = r"data\basler_0409\20260403_154225"


START_SEC = 10       # 시작 시간 (초)
DURATION_SEC = 5    # 재생 길이 (초) (None 또는 0으로 설정 시 끝까지 재생)
VIEW_SCALE = 0.6     # 화면 크기 비율 (0.5 = 50%)
# =========================================================

def load_settings_from_txt(target_dir):
    """
    camera_all_settings.txt 파일 상단의 JSON 요약본을 파싱하여 
    실제 녹화 시간, 설정 FPS, 픽셀 포맷을 반환합니다.
    """
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_duration = 60.0
    default_fps = 60.0
    default_format = "BayerRG12"
    
    if not os.path.exists(settings_path):
        print(f"⚠️ 'camera_all_settings.txt'가 없습니다. 기본값으로 재생합니다.")
        return default_duration, default_fps, default_format

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            
            duration = summary.get("Record_Duration_sec", default_duration)
            fps = summary.get("FPS_Result", summary.get("FPS_Target", default_fps))
            pixel_format = summary.get("PixelFormat", default_format)
            
            print(f"📄 [설정 파일 로드 성공]")
            print(f"   - 총 녹화 시간: {duration} 초")
            print(f"   - 녹화 FPS    : {fps} fps")
            print(f"   - 픽셀 포맷   : {pixel_format}")
            return float(duration), float(fps), pixel_format
            
    except Exception as e:
        print(f"⚠️ 설정 파일 파싱 실패 ({e}). 기본값으로 진행합니다.")
        
    return default_duration, default_fps, default_format


def play_synced_data():
    if not os.path.exists(TARGET_DIR):
        print("❌ 폴더를 찾을 수 없습니다.")
        return

    print("📂 데이터 로딩 및 동기화 준비 중...")

    # 💡 1. 텍스트 파일에서 동적 파라미터 로드 (하드코딩 제거)
    total_duration, play_fps, pixel_format = load_settings_from_txt(TARGET_DIR)

    # 2. CSV 로드
    cam_csv = os.path.join(TARGET_DIR, "camera_timestamps.csv")
    sen_csv = os.path.join(TARGET_DIR, "ppg_sensor.csv")
    
    if not os.path.exists(cam_csv) or not os.path.exists(sen_csv):
        print("❌ CSV 파일이 없습니다.")
        return

    df_cam = pd.read_csv(cam_csv)
    df_sen = pd.read_csv(sen_csv)

    # 3. 시간 동기화
    base_time = df_cam['Timestamp'].iloc[0]
    df_cam['Rel_Time'] = df_cam['Timestamp'] - base_time
    df_sen['Rel_Time'] = df_sen['Timestamp'] - base_time

    # 4. 구간 자르기 (DURATION_SEC이 없으면 전체 재생)
    end_sec = START_SEC + DURATION_SEC if DURATION_SEC else total_duration
    
    mask_cam = (df_cam['Rel_Time'] >= START_SEC) & (df_cam['Rel_Time'] <= end_sec)
    df_cam_cut = df_cam[mask_cam].reset_index(drop=True)
    
    mask_sen = (df_sen['Rel_Time'] >= START_SEC) & (df_sen['Rel_Time'] <= end_sec)
    df_sen_cut = df_sen[mask_sen].reset_index(drop=True)

    if len(df_cam_cut) == 0:
        print("⚠️ 해당 구간에 데이터가 없습니다.")
        return
    
    frames_dir = os.path.join(TARGET_DIR, "frames")

    # 이미지 크기 자동 감지 (그래프 width 맞추기 위해)
    first_fname = f"frame_{int(df_cam_cut.iloc[0]['Frame_Index']):04d}.tiff"
    first_path = os.path.join(frames_dir, first_fname)
    if os.path.exists(first_path):
        tmp_img = tifffile.imread(first_path)
        IMG_H, IMG_W = tmp_img.shape[:2]
    else:
        IMG_W = 1024 # 파일 없으면 기본값

    print(f"🎬 재생 시작: {START_SEC}~{end_sec}초 (원본: {IMG_W}x{IMG_H} -> 축소: {int(VIEW_SCALE*100)}%)")
    print("   [Space]: 일시정지 / [ESC]: 종료")

    # 5. 그래프 배경 그리기
    W_IMG = IMG_W  
    H_GRAPH = 200 
    graph_bg = np.zeros((H_GRAPH, W_IMG, 3), dtype=np.uint8) 

    if len(df_sen_cut) > 1:
        t_vals = df_sen_cut['Rel_Time'].values
        
        # 컬럼 이름 호환성 체크 (Raw vs Processed)
        col_name = 'IR_Value_Raw' if 'IR_Value_Raw' in df_sen_cut.columns else 'IR_Value'
        y_vals = df_sen_cut[col_name].values
        
        # X축/Y축 스케일링
        x_norm = (t_vals - START_SEC) / (end_sec - START_SEC) * W_IMG
        
        y_min, y_max = y_vals.min(), y_vals.max()
        if y_max == y_min: y_max += 1 
        
        y_norm = H_GRAPH - ((y_vals - y_min) / (y_max - y_min) * (H_GRAPH - 20)) - 10

        pts = np.column_stack((x_norm, y_norm)).astype(np.int32)
        cv2.polylines(graph_bg, [pts], isClosed=False, color=(0, 255, 0), thickness=2)

    # 6. 재생 루프
    wait_ms = int(1000 / play_fps) if play_fps > 0 else 16 # 💡 동적 재생 속도 적용

    for i, row in df_cam_cut.iterrows():
        # --- 영상 로드 ---
        fname = f"frame_{int(row['Frame_Index']):04d}.tiff"
        path = os.path.join(frames_dir, fname)
        
        if not os.path.exists(path):
            continue

        raw_img = tifffile.imread(path)
        
        # 💡 [수정] 픽셀 포맷에 따른 동적 이미지 변환 로직 적용
        if pixel_format == "BGR8":
            view_img = raw_img  # 이미 8-bit BGR 이므로 그대로 사용
        elif pixel_format == "BayerRG8":
            view_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
        else: # 기본 BayerRG12
            rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
            view_img = (rgb_img >> 4).astype(np.uint8) # 12bit -> 8bit 변환

        # --- 그래프 업데이트 ---
        graph_curr = graph_bg.copy()
        curr_t = row['Rel_Time']
        
        # 커서 위치 계산 (비율에 맞춰 정확히)
        cursor_x = int((curr_t - START_SEC) / (end_sec - START_SEC) * W_IMG)
        
        # 커서 그리기
        cv2.line(graph_curr, (cursor_x, 0), (cursor_x, H_GRAPH), (0, 0, 255), 2)
        cv2.putText(graph_curr, f"T: {curr_t:.3f}s", (10, 30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # --- 합치기 ---
        try:
            final_view = cv2.vconcat([view_img, graph_curr])
        except:
            print("⚠️ 이미지와 그래프 크기가 맞지 않아 합칠 수 없습니다.")
            final_view = view_img

        # --- 리사이즈 (보기 편하게) ---
        h, w = final_view.shape[:2]
        new_w = int(w * VIEW_SCALE)
        new_h = int(h * VIEW_SCALE)
        final_view = cv2.resize(final_view, (new_w, new_h), interpolation=cv2.INTER_AREA)

        cv2.imshow("Sync Player", final_view)

        key = cv2.waitKey(wait_ms)
        if key == 27: # ESC
            break
        elif key == 32: # Space
            cv2.waitKey()

    cv2.destroyAllWindows()
    print("✨ 종료되었습니다.")

if __name__ == "__main__":
    play_synced_data()

PPG만(피크검출)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
import os

# =========================================================
# [설정] 분석할 폴더 및 옵션
# =========================================================
# 분석할 데이터 폴더 경로 (ppg_sensor.csv가 있는 곳)
TARGET_DIR = r"data\basler_0402\20260402_193744"
TARGET_DIR = r"data\basler_0402\20260402_194405"

START_SEC = 10       # 분석 시작 시간 (초)
DURATION_SEC = 5     # 분석할 길이 (초)

# 대상 설정 ('human' 또는 'mouse')
# - human: 0.5~5Hz 필터, 피크 간격 0.3초 이상
# - mouse: 2.0~15Hz 필터, 피크 간격 0.05초 이상
TARGET_TYPE = 'human' 
# TARGET_TYPE = 'mouse' 

# 그래프 설정
INVERT_SIGNAL = False  # 신호가 뒤집혀 보이면 True로 변경 (수축기 피크를 위로)
SHOW_FILTERED = True   # 필터링된 신호로 계산할지 여부
# =========================================================

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

def analyze_ppg():
    csv_path = os.path.join(TARGET_DIR, "ppg_sensor.csv")
    
    if not os.path.exists(csv_path):
        print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")
        return

    print(f"📂 데이터 로드 중: {csv_path}")
    df = pd.read_csv(csv_path)

    # 1. 컬럼 이름 확인 및 데이터 추출
    # (저장 방식에 따라 컬럼명이 다를 수 있어 처리)
    if 'Timestamp' in df.columns:
        t_raw = df['Timestamp'].values
    elif 'Time' in df.columns:
        t_raw = df['Time'].values
    else:
        # 타임스탬프가 없으면 인덱스 기반 생성 (200Hz 가정)
        t_raw = np.arange(len(df)) / 200.0

    if 'IR_Value_Raw' in df.columns:
        y_raw = df['IR_Value_Raw'].values
    elif 'IR_Value' in df.columns:
        y_raw = df['IR_Value'].values
    else:
        print("❌ PPG 데이터 컬럼을 찾을 수 없습니다.")
        return

    # 2. 시간축 정규화 (0초부터 시작하도록)
    t_rel = t_raw - t_raw[0]

    # 3. 구간 자르기
    end_sec = START_SEC + DURATION_SEC
    mask = (t_rel >= START_SEC) & (t_rel <= end_sec)
    
    t_cut = t_rel[mask]
    y_cut = y_raw[mask]

    if len(t_cut) == 0:
        print("⚠️ 해당 구간에 데이터가 없습니다.")
        return

    # 4. 샘플링 레이트(Fs) 자동 계산
    # (실제 데이터의 평균 시간 간격으로 계산)
    dt_mean = np.mean(np.diff(t_cut))
    fs_real = 1.0 / dt_mean
    print(f"⚡ 추정 샘플링 레이트: {fs_real:.2f} Hz")

    # 5. 전처리 (필터링)
    # DC 성분(기저선) 제거 및 고주파 노이즈 제거
    if TARGET_TYPE == 'mouse':
        low_cut, high_cut = 2.0, 15.0  # 쥐: 빠른 심박 (120~900 BPM 커버)
        min_dist_sec = 0.05            # 최소 피크 간격 (50ms)
    else:
        low_cut, high_cut = 0.5, 5.0   # 사람: 일반 심박 (30~300 BPM 커버)
        min_dist_sec = 0.35            # 최소 피크 간격 (350ms)

    # 옵션: 신호 반전 (PPG는 흡수율이라 피크가 아래로 갈 수 있음)
    if INVERT_SIGNAL:
        y_cut = -y_cut

    # 필터 적용
    y_filtered = butter_bandpass_filter(y_cut, low_cut, high_cut, fs_real, order=4)
    
    # 분석 대상 신호 선택
    y_target = y_filtered if SHOW_FILTERED else y_cut

    # 6. 피크 검출
    # distance: 피크 간 최소 거리 (샘플 수)
    distance_samples = int(min_dist_sec * fs_real)
    
    # height: 최소 높이 (평균보다 높아야 함)
    # prominence: 주변보다 얼마나 튀어나왔는지 (매우 중요)
    peaks, properties = find_peaks(y_target, distance=distance_samples, prominence=np.std(y_target)*0.5)

    # 7. BPM 계산
    if len(peaks) > 1:
        peak_times = t_cut[peaks]
        ibi_intervals = np.diff(peak_times) # 피크 간 시간 차이 (초)
        mean_ibi = np.mean(ibi_intervals)
        bpm = 60.0 / mean_ibi
        bpm_text = f"BPM: {bpm:.1f}"
    else:
        bpm = 0
        bpm_text = "BPM: -- (피크 부족)"

    print(f"❤️ 결과: {bpm_text} (검출된 피크 수: {len(peaks)})")

    # =========================================================
    # [시각화] Matplotlib 그래프 그리기
    # =========================================================
    plt.figure(figsize=(12, 6))
    
    # (1) 원본 신호 (흐리게)
    # 스케일을 맞추기 위해 필터된 신호 평균에 맞춰서 이동만 시킴 (비교용)
    plt.plot(t_cut, y_cut - np.mean(y_cut), color='gray', alpha=0.3, label='Raw Signal (Centered)')
    
    # (2) 필터링된 신호 (진하게)
    plt.plot(t_cut, y_target, color='#1f77b4', linewidth=2, label='Filtered Signal')

    # (3) 검출된 피크 표시
    if len(peaks) > 0:
        plt.plot(t_cut[peaks], y_target[peaks], "x", color='red', markersize=10, markeredgewidth=3, label='Detected Peaks')
        
        # 피크마다 선 긋기 (시각적 도움)
        for p in peaks:
            plt.axvline(x=t_cut[p], color='red', alpha=0.1, linestyle='--')

    plt.title(f"PPG Analysis ({START_SEC}~{START_SEC+DURATION_SEC}s) | {bpm_text}", fontsize=16, fontweight='bold')
    plt.xlabel("Time (seconds)", fontsize=12)
    plt.ylabel("Amplitude (Normalized)", fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    plt.show()

if __name__ == "__main__":
    analyze_ppg()

프레임 ONLY , 채널 선택

In [ ]:
# 파일명: analyze_single_channel_absolute_raw.py
# 지시사항: 지정된 시간 구간 및 단일 채널(R/G/B)의 프레임을 분석합니다. 상단 Raw 그래프는 8비트/12비트 센서의 실제 절대 픽셀값(평균을 빼지 않음)을 Y축에 그대로 출력하고, 중간에는 필터링된 신호를, 하단에는 FFT 기반 파워 스펙트럼(BPM 단위)을 출력합니다.

import os
import cv2
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
import json

# =========================================================
# [설정] 분석할 폴더 및 옵션
# =========================================================
TARGET_DIR = r"data\basler_0409\20260403_153112"
TARGET_DIR = r"data\basler_0409\20260403_153547"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_153712"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_154017"    #발바닥, 카메라 흔들림
TARGET_DIR = r"data\basler_0409\20260403_154225"    #발바닥 *
# TARGET_DIR = r"data\basler_0409\20260403_154512"
# TARGET_DIR = r"data\basler_0409\20260403_154645"    # 카메라 흔들림
# TARGET_DIR = r"data\basler_0409\20260403_155010"    # 근접
# TARGET_DIR = r"data\basler_0409\20260403_155121"
# TARGET_DIR = r"data\basler_0409\20260403_155212"    #근접
# TARGET_DIR = r"data\basler_0409\20260403_155430"    # 근접, 꼬리 , 발바닥
# TARGET_DIR = r"data\basler_0409\20260403_155624"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_155725"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_155936"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_161750"    #비마취
# TARGET_DIR = r"data\basler_0409\20260403_161958"
# TARGET_DIR = r"data\basler_0409\20260403_162203"
# TARGET_DIR = r"data\basler_0409\20260403_162415"
# TARGET_DIR = r"data\basler_0409\20260403_162607"

FRAMES_SUBDIR = "frames" 

# 분석할 단일 채널 선택 ('R', 'G', 'B' 중 택 1)
TARGET_CHANNEL = 'R'  

# 분석 시간 구간 설정
START_SEC = 0       # 분석 시작 시간 (초)
DURATION_SEC = 20   # 분석할 길이 (초)

# 💡 밴드 패스 필터 범위 설정 (BPM 단위)
FILTER_BPM_MIN = 20   # 필터 하한 (BPM)
FILTER_BPM_MAX = 40   # 필터 상한 (BPM)

# 그래프 설정
INVERT_CAMERA = False  # 카메라 신호 뒤집기
# =========================================================

def load_camera_settings(target_dir):
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_format = "BayerRG12"
    default_fps = 60.0
    
    if not os.path.exists(settings_path):
        print(f"⚠️ 'camera_all_settings.txt'가 없습니다. 기본값(Format: {default_format}, FPS: {default_fps})으로 진행합니다.")
        return default_format, default_fps

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            
            pixel_format = summary.get("PixelFormat", default_format)
            fps = summary.get("FPS_Result", summary.get("FPS_Target", default_fps))
            
            print(f"📄 [설정 파일 로드] 픽셀 포맷: {pixel_format} | 🎥 카메라 FPS: {fps:.2f}")
            return pixel_format, float(fps)
            
    except Exception as e:
        print(f"⚠️ 설정 파싱 실패 ({e}). 기본값으로 진행합니다.")
        
    return default_format, default_fps

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

def analyze_single_channel_from_frames():
    search_dir = os.path.join(TARGET_DIR, FRAMES_SUBDIR)
    if not os.path.exists(search_dir):
        search_dir = TARGET_DIR

    frame_files = sorted([f for f in os.listdir(search_dir) if f.lower().endswith(('.tiff', '.tif', '.png', '.jpg'))])
    
    if not frame_files:
        print(f"❌ '{search_dir}' 경로에서 이미지 파일을 찾을 수 없습니다.")
        return

    pixel_format, camera_fps = load_camera_settings(TARGET_DIR)

    # =========================================================
    # [시간 구간 분할] 
    # =========================================================
    total_frames = len(frame_files)
    start_idx = int(START_SEC * camera_fps)
    end_idx = int((START_SEC + DURATION_SEC) * camera_fps)
    
    if start_idx >= total_frames:
        print(f"❌ 시작 시간이 영상 길이를 벗어났습니다.")
        return
        
    end_idx = min(end_idx, total_frames)
    target_frame_files = frame_files[start_idx:end_idx]
    
    print(f"✂️ [구간 설정] {START_SEC}초 ~ {START_SEC+DURATION_SEC}초 (프레임: {start_idx} ~ {end_idx-1})")

    # =========================================================
    # [1단계] ROI 설정
    # =========================================================
    first_frame_path = os.path.join(search_dir, target_frame_files[0])
    
    if first_frame_path.lower().endswith(('.tiff', '.tif')):
        raw_img = tifffile.imread(first_frame_path)
    else:
        raw_img = cv2.imread(first_frame_path, cv2.IMREAD_UNCHANGED)

    if len(raw_img.shape) == 3 and raw_img.shape[2] == 3:
        view_img = raw_img
    elif pixel_format == "BGR8":
        view_img = raw_img
    elif pixel_format == "BayerRG8":
        view_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
    else: 
        rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
        view_img = (rgb_img >> 4).astype(np.uint8)

    cv2.namedWindow("Select ROI (Drag & Press SPACE)", cv2.WINDOW_NORMAL)
    roi = cv2.selectROI("Select ROI (Drag & Press SPACE)", view_img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

    x, y, w, h = [int(v) for v in roi]
    
    if w == 0 or h == 0:
        print("⚠️ ROI가 선택되지 않았습니다. 종료합니다.")
        return
        
    x, y = x & ~1, y & ~1

    # =========================================================
    # [2단계] 타겟 프레임에서 선택한 단일 채널만 추출
    # =========================================================
    print(f"⏳ [2/3] {len(target_frame_files)}개 프레임에서 '{TARGET_CHANNEL}' 채널 값을 추출 중입니다...")
    
    channel_idx_map = {'B': 0, 'G': 1, 'R': 2}
    ch_idx = channel_idx_map.get(TARGET_CHANNEL.upper(), 1)
    
    raw_vals = []
    
    for f_name in target_frame_files:
        f_path = os.path.join(search_dir, f_name)
        
        if f_name.lower().endswith(('.tiff', '.tif')):
            img = tifffile.imread(f_path)
        else:
            img = cv2.imread(f_path, cv2.IMREAD_UNCHANGED)
        
        crop_img = img[y:y+h, x:x+w]
        
        if len(crop_img.shape) == 3 and crop_img.shape[2] == 3:
            bgr_crop = crop_img
        elif pixel_format == "BGR8":
            bgr_crop = crop_img
        else:
            bgr_crop = cv2.cvtColor(crop_img, cv2.COLOR_BayerBG2BGR)
        
        raw_vals.append(np.mean(bgr_crop[:, :, ch_idx]))

    y_raw = np.array(raw_vals)
    t_cam = np.arange(start_idx, start_idx + len(target_frame_files)) / camera_fps

    # =========================================================
    # [3단계] 신호 필터링 및 피크 검출
    # =========================================================
    print(f"⚡ [3/3] 신호를 필터링하고 주파수를 분석합니다...")

    # 💡 BPM을 Hz로 변환하여 밴드 패스 필터 주파수 설정
    low_cut = FILTER_BPM_MIN / 60.0
    high_cut = FILTER_BPM_MAX / 60.0
    min_dist = (60.0 / FILTER_BPM_MAX) * 0.5  # 최대 BPM 기준 간격의 절반

    y_raw_centered = y_raw - np.mean(y_raw)
    if INVERT_CAMERA:
        y_raw_centered = -y_raw_centered

    # 신호 필터링
    y_filt = butter_bandpass_filter(y_raw_centered, low_cut, high_cut, camera_fps)
    peaks, _ = find_peaks(y_filt, distance=int(min_dist * camera_fps), prominence=np.std(y_filt)*0.5)

    bpm_peak = 60.0 / np.mean(np.diff(t_cam[peaks])) if len(peaks) > 1 else 0

    # =========================================================
    # [4단계] FFT(고속 푸리에 변환)를 통한 파워 스펙트럼 계산
    # =========================================================
    n = len(y_filt)
    dt = 1.0 / camera_fps
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(y_filt)

    power = np.abs(fft_vals) ** 2

    # 양수 주파수 영역만 필터링 (DC 성분 0Hz 제외)
    pos_mask = freqs > 0
    pos_freqs_bpm = freqs[pos_mask] * 60.0
    pos_power = power[pos_mask]

    if len(pos_power) > 0:
        max_idx = np.argmax(pos_power)
        bpm_fft = pos_freqs_bpm[max_idx]
    else:
        bpm_fft = 0

    print(f"❤️ 측정 결과 -> Peak 기반 BPM: {bpm_peak:.1f} | FFT 기반 우세 BPM: {bpm_fft:.1f}")

    # =========================================================
    # [시각화] 상단: Raw Data / 중간: Filtered Data / 하단: FFT 파워 스펙트럼
    # =========================================================
    # X축 공유는 상단, 중간 그래프에만 적용하고 FFT는 독립된 X축(BPM) 사용
    fig = plt.figure(figsize=(14, 12))
    
    color_map = {'R': '#ef4444', 'G': '#22c55e', 'B': '#3b82f6'}
    ch_color = color_map.get(TARGET_CHANNEL.upper(), '#22c55e')
    ch_name = {'R': 'Red', 'G': 'Green', 'B': 'Blue'}.get(TARGET_CHANNEL.upper(), 'Green')

    # --- [상단] Raw 신호 그래프 ---
    ax_raw = plt.subplot(3, 1, 1)
    ax_raw.plot(t_cam, y_raw, color='gray', alpha=0.8, linewidth=1.5, label=f'Absolute Raw {ch_name}')
    ax_raw.set_title(f"[{ch_name} Channel] Absolute Raw Pixel Value", fontsize=14, fontweight='bold')
    ax_raw.set_ylabel("Pixel Intensity (Amplitude)")
    ax_raw.legend(loc='upper right')
    ax_raw.grid(True, linestyle='--', alpha=0.6)

    # --- [중간] 필터링 신호 및 피크 그래프 ---
    ax_filt = plt.subplot(3, 1, 2, sharex=ax_raw)
    ax_filt.plot(t_cam, y_filt, color=ch_color, linewidth=2, label=f'Filtered {ch_name}')
    if len(peaks) > 0:
        ax_filt.plot(t_cam[peaks], y_filt[peaks], "x", color='red', markersize=8, markeredgewidth=2, label='Peaks')
    ax_filt.set_title(f"[{ch_name} Channel] Filtered AC Signal | Peak BPM: {bpm_peak:.1f}", fontsize=14, fontweight='bold', color=ch_color)
    ax_filt.set_xlabel("Time (seconds)", fontsize=12)
    ax_filt.set_ylabel("Amplitude (Zero-Mean)")
    ax_filt.legend(loc='upper right')
    ax_filt.grid(True, linestyle='--', alpha=0.6)

    # --- [하단] 💡 FFT 파워 스펙트럼 그래프 ---
    ax_fft = plt.subplot(3, 1, 3)
    ax_fft.plot(pos_freqs_bpm, pos_power, color='purple', linewidth=1.5, label='FFT Power')
    
    # 설정한 필터 밴드 범위 마킹
    ax_fft.axvline(x=FILTER_BPM_MIN, color='black', linestyle=':', alpha=0.5, label='Filter Min')
    ax_fft.axvline(x=FILTER_BPM_MAX, color='black', linestyle=':', alpha=0.5, label='Filter Max')
    
    # 우세 주파수 마커 표시
    if len(pos_power) > 0:
        ax_fft.plot(bpm_fft, pos_power[max_idx], "o", color='red', markersize=6, label=f'Dominant: {bpm_fft:.1f} BPM')

    ax_fft.set_title(f"[{ch_name} Channel] FFT Power Spectrum | FFT BPM: {bpm_fft:.1f}", fontsize=14, fontweight='bold', color='purple')
    ax_fft.set_xlabel("Frequency (BPM)", fontsize=12)
    ax_fft.set_ylabel("Power")
    
    # X축 범위를 설정한 BPM 기준 주변으로 여유 있게 설정
    pad_bpm = (FILTER_BPM_MAX - FILTER_BPM_MIN) * 0.2
    ax_fft.set_xlim(max(0, FILTER_BPM_MIN - pad_bpm), FILTER_BPM_MAX + pad_bpm)
    ax_fft.legend(loc='upper right')
    ax_fft.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_single_channel_from_frames()

프레임 ONLY, 평균대신 중앙값 사용  

In [ ]:
# 파일명: analyze_single_channel_absolute_raw.py
# 지시사항: 지정된 시간 구간 및 단일 채널(R/G/B)의 프레임을 분석합니다. 상단 Raw 그래프는 8비트/12비트 센서의 실제 절대 픽셀값(평균을 빼지 않음)을 Y축에 그대로 출력하고, 하단에는 필터링된 신호를 출력합니다.
# [수정사항] 이상치(조명 반사, 털) 제거를 위해 ROI 내 픽셀의 단순 평균(Mean) 대신 중앙값(Median)을 추출하도록 변경되었습니다.

import os
import cv2
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
import json

# =========================================================
# [설정] 분석할 폴더 및 옵션
# =========================================================
TARGET_DIR = r"data\basler_0409\20260403_153112"
TARGET_DIR = r"data\basler_0409\20260403_153547"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_153712"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_154017"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_154225"    #발바닥 *
# TARGET_DIR = r"data\basler_0409\20260403_154512"
# TARGET_DIR = r"data\basler_0409\20260403_154645"    # 배경에 신호
TARGET_DIR = r"data\basler_0409\20260403_155010"    # 근접
# TARGET_DIR = r"data\basler_0409\20260403_155121"
# TARGET_DIR = r"data\basler_0409\20260403_155212"
# TARGET_DIR = r"data\basler_0409\20260403_155430"    # 근접, 꼬리 , 발바닥
TARGET_DIR = r"data\basler_0409\20260403_155624"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_155725"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_155936"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_161750"    #비마취
TARGET_DIR = r"data\basler_0409\20260403_161958"
# TARGET_DIR = r"data\basler_0409\20260403_162203"
# TARGET_DIR = r"data\basler_0409\20260403_162415"
# TARGET_DIR = r"data\basler_0409\20260403_162607"

FRAMES_SUBDIR = "frames" 

# 분석할 단일 채널 선택 ('R', 'G', 'B' 중 택 1)
TARGET_CHANNEL = 'R'  

# 분석 시간 구간 설정
START_SEC = 3       # 분석 시작 시간 (초)
DURATION_SEC = 10   # 분석할 길이 (초)

# 💡 밴드 패스 필터 범위 설정 (BPM 단위)
FILTER_BPM_MIN = 120   # 필터 하한 (BPM)
FILTER_BPM_MAX = 200   # 필터 상한 (BPM)

# 그래프 설정
INVERT_CAMERA = False  # 카메라 신호 뒤집기
# =========================================================

def load_camera_settings(target_dir):
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_format = "BayerRG12"
    default_fps = 60.0
    
    if not os.path.exists(settings_path):
        print(f"⚠️ 'camera_all_settings.txt'가 없습니다. 기본값(Format: {default_format}, FPS: {default_fps})으로 진행합니다.")
        return default_format, default_fps

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            
            pixel_format = summary.get("PixelFormat", default_format)
            fps = summary.get("FPS_Result", summary.get("FPS_Target", default_fps))
            
            print(f"📄 [설정 파일 로드] 픽셀 포맷: {pixel_format} | 🎥 카메라 FPS: {fps:.2f}")
            return pixel_format, float(fps)
            
    except Exception as e:
        print(f"⚠️ 설정 파싱 실패 ({e}). 기본값으로 진행합니다.")
        
    return default_format, default_fps

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

def analyze_single_channel_from_frames():
    search_dir = os.path.join(TARGET_DIR, FRAMES_SUBDIR)
    if not os.path.exists(search_dir):
        search_dir = TARGET_DIR

    frame_files = sorted([f for f in os.listdir(search_dir) if f.lower().endswith(('.tiff', '.tif', '.png', '.jpg'))])
    
    if not frame_files:
        print(f"❌ '{search_dir}' 경로에서 이미지 파일을 찾을 수 없습니다.")
        return

    pixel_format, camera_fps = load_camera_settings(TARGET_DIR)

    # =========================================================
    # [시간 구간 분할] 
    # =========================================================
    total_frames = len(frame_files)
    start_idx = int(START_SEC * camera_fps)
    end_idx = int((START_SEC + DURATION_SEC) * camera_fps)
    
    if start_idx >= total_frames:
        print(f"❌ 시작 시간이 영상 길이를 벗어났습니다.")
        return
        
    end_idx = min(end_idx, total_frames)
    target_frame_files = frame_files[start_idx:end_idx]
    
    print(f"✂️ [구간 설정] {START_SEC}초 ~ {START_SEC+DURATION_SEC}초 (프레임: {start_idx} ~ {end_idx-1})")

    # =========================================================
    # [1단계] ROI 설정
    # =========================================================
    first_frame_path = os.path.join(search_dir, target_frame_files[0])
    
    if first_frame_path.lower().endswith(('.tiff', '.tif')):
        raw_img = tifffile.imread(first_frame_path)
    else:
        raw_img = cv2.imread(first_frame_path, cv2.IMREAD_UNCHANGED)

    if len(raw_img.shape) == 3 and raw_img.shape[2] == 3:
        view_img = raw_img
    elif pixel_format == "BGR8":
        view_img = raw_img
    elif pixel_format == "BayerRG8":
        view_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
    else: 
        rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
        view_img = (rgb_img >> 4).astype(np.uint8)

    cv2.namedWindow("Select ROI (Drag & Press SPACE)", cv2.WINDOW_NORMAL)
    roi = cv2.selectROI("Select ROI (Drag & Press SPACE)", view_img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

    x, y, w, h = [int(v) for v in roi]
    
    if w == 0 or h == 0:
        print("⚠️ ROI가 선택되지 않았습니다. 종료합니다.")
        return
        
    x, y = x & ~1, y & ~1

    # =========================================================
    # [2단계] 타겟 프레임에서 선택한 단일 채널만 추출
    # =========================================================
    print(f"⏳ [2/3] {len(target_frame_files)}개 프레임에서 '{TARGET_CHANNEL}' 채널 중앙값을 추출 중입니다...")
    
    channel_idx_map = {'B': 0, 'G': 1, 'R': 2}
    ch_idx = channel_idx_map.get(TARGET_CHANNEL.upper(), 1)
    
    raw_vals = []
    
    for f_name in target_frame_files:
        f_path = os.path.join(search_dir, f_name)
        
        if f_name.lower().endswith(('.tiff', '.tif')):
            img = tifffile.imread(f_path)
        else:
            img = cv2.imread(f_path, cv2.IMREAD_UNCHANGED)
        
        crop_img = img[y:y+h, x:x+w]
        
        if len(crop_img.shape) == 3 and crop_img.shape[2] == 3:
            bgr_crop = crop_img
        elif pixel_format == "BGR8":
            bgr_crop = crop_img
        else:
            bgr_crop = cv2.cvtColor(crop_img, cv2.COLOR_BayerBG2BGR)
        
        # 💡 공간적 노이즈(스펙큘러, 그림자, 털 등) 제거를 위해 np.mean 대신 np.median 사용
        raw_vals.append(np.median(bgr_crop[:, :, ch_idx]))

    y_raw = np.array(raw_vals)
    t_cam = np.arange(start_idx, start_idx + len(target_frame_files)) / camera_fps

    # =========================================================
    # [3단계] 신호 필터링 및 피크 검출
    # =========================================================
    print(f"⚡ [3/3] 신호를 필터링하고 주파수를 분석합니다...")

    # BPM을 Hz로 변환하여 밴드 패스 필터 주파수 설정
    low_cut = FILTER_BPM_MIN / 60.0
    high_cut = FILTER_BPM_MAX / 60.0
    min_dist = (60.0 / FILTER_BPM_MAX) * 0.5  # 최대 BPM 기준 간격의 절반

    # 필터링 전용으로 평균을 뺀 신호(Zero-mean)를 사용하여 필터 왜곡 방지
    y_raw_centered = y_raw - np.mean(y_raw)
    if INVERT_CAMERA:
        y_raw_centered = -y_raw_centered

    # 신호 필터링
    y_filt = butter_bandpass_filter(y_raw_centered, low_cut, high_cut, camera_fps)
    peaks, _ = find_peaks(y_filt, distance=int(min_dist * camera_fps), prominence=np.std(y_filt)*0.5)

    bpm_peak = 60.0 / np.mean(np.diff(t_cam[peaks])) if len(peaks) > 1 else 0

    # =========================================================
    # [4단계] FFT(고속 푸리에 변환)를 통한 파워 스펙트럼 계산
    # =========================================================
    n = len(y_filt)
    dt = 1.0 / camera_fps
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(y_filt)

    power = np.abs(fft_vals) ** 2

    # 양수 주파수 영역만 필터링 (DC 성분 0Hz 제외)
    pos_mask = freqs > 0
    pos_freqs_bpm = freqs[pos_mask] * 60.0
    pos_power = power[pos_mask]

    if len(pos_power) > 0:
        max_idx = np.argmax(pos_power)
        bpm_fft = pos_freqs_bpm[max_idx]
    else:
        bpm_fft = 0

    print(f"❤️ 측정 결과 -> Peak 기반 BPM: {bpm_peak:.1f} | FFT 기반 우세 BPM: {bpm_fft:.1f}")

    # =========================================================
    # [시각화] 상단: Raw Data / 중간: Filtered Data / 하단: FFT 파워 스펙트럼
    # =========================================================
    fig = plt.figure(figsize=(14, 12))
    
    color_map = {'R': '#ef4444', 'G': '#22c55e', 'B': '#3b82f6'}
    ch_color = color_map.get(TARGET_CHANNEL.upper(), '#22c55e')
    ch_name = {'R': 'Red', 'G': 'Green', 'B': 'Blue'}.get(TARGET_CHANNEL.upper(), 'Green')

    # --- [상단] Raw 신호 그래프 ---
    ax_raw = plt.subplot(3, 1, 1)
    ax_raw.plot(t_cam, y_raw, color='gray', alpha=0.8, linewidth=1.5, label=f'Absolute Raw {ch_name} (Median)')
    ax_raw.set_title(f"[{ch_name} Channel] Absolute Raw Pixel Value (Spatial Median)", fontsize=14, fontweight='bold')
    ax_raw.set_ylabel("Pixel Intensity (Amplitude)")
    ax_raw.legend(loc='upper right')
    ax_raw.grid(True, linestyle='--', alpha=0.6)

    # --- [중간] 필터링 신호 및 피크 그래프 ---
    ax_filt = plt.subplot(3, 1, 2, sharex=ax_raw)
    ax_filt.plot(t_cam, y_filt, color=ch_color, linewidth=2, label=f'Filtered {ch_name}')
    if len(peaks) > 0:
        ax_filt.plot(t_cam[peaks], y_filt[peaks], "x", color='red', markersize=8, markeredgewidth=2, label='Peaks')
    ax_filt.set_title(f"[{ch_name} Channel] Filtered AC Signal | Peak BPM: {bpm_peak:.1f}", fontsize=14, fontweight='bold', color=ch_color)
    ax_filt.set_xlabel("Time (seconds)", fontsize=12)
    ax_filt.set_ylabel("Amplitude (Zero-Mean)")
    ax_filt.legend(loc='upper right')
    ax_filt.grid(True, linestyle='--', alpha=0.6)

    # --- [하단] FFT 파워 스펙트럼 그래프 ---
    ax_fft = plt.subplot(3, 1, 3)
    ax_fft.plot(pos_freqs_bpm, pos_power, color='purple', linewidth=1.5, label='FFT Power')
    
    ax_fft.axvline(x=FILTER_BPM_MIN, color='black', linestyle=':', alpha=0.5, label='Filter Min')
    ax_fft.axvline(x=FILTER_BPM_MAX, color='black', linestyle=':', alpha=0.5, label='Filter Max')
    
    if len(pos_power) > 0:
        ax_fft.plot(bpm_fft, pos_power[max_idx], "o", color='red', markersize=6, label=f'Dominant: {bpm_fft:.1f} BPM')

    ax_fft.set_title(f"[{ch_name} Channel] FFT Power Spectrum | FFT BPM: {bpm_fft:.1f}", fontsize=14, fontweight='bold', color='purple')
    ax_fft.set_xlabel("Frequency (BPM)", fontsize=12)
    ax_fft.set_ylabel("Power")
    
    pad_bpm = (FILTER_BPM_MAX - FILTER_BPM_MIN) * 0.2
    ax_fft.set_xlim(max(0, FILTER_BPM_MIN - pad_bpm), FILTER_BPM_MAX + pad_bpm)
    ax_fft.legend(loc='upper right')
    ax_fft.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_single_channel_from_frames()

지정영역 내 R,G,B 채널 평균, PPG데이터 같이 출력

In [ ]:
# 파일명: analyze_ppg_and_camera_dynamic_channel.py
# 지시사항: 하드웨어 PPG 센서 데이터와 카메라 rPPG 데이터를 함께 분석합니다. 
# 사용자가 설정한 카메라 채널(R, G, B)에 따라 데이터를 추출하고, 2x2 배열 그래프(상단: Raw 절대값, 하단: Filtered 신호)로 시각화합니다.

import os
import cv2
import tifffile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
import json

# =========================================================
# [설정] 분석할 폴더 및 옵션
# =========================================================
TARGET_DIR = r"data\basler_0409\20260403_161750"
TARGET_DIR = r"data\basler_0409\20260403_161958"
TARGET_DIR = r"data\basler_0409\20260403_162203"
# TARGET_DIR = r"data\basler_0409\20260403_162415"
# TARGET_DIR = r"data\basler_0409\20260403_162607"

START_SEC = 10       # 분석 시작 시간 (초)
DURATION_SEC = 5    # 분석할 길이 (초)

# 💡 카메라 분석 채널 설정 ('R', 'G', 'B' 중 택 1)
TARGET_CHANNEL = 'R'

# 대상 설정 ('human' 또는 'mouse')
TARGET_TYPE = 'mouse' 

# 그래프 설정
INVERT_SENSOR = False  # 센서 신호 뒤집기
INVERT_CAMERA = False  # 카메라 신호 뒤집기 (혈류량 비례를 위해 G채널은 주로 True 사용)
SHOW_FILTERED = True   # 필터링된 신호로 피크를 계산할지 여부
# =========================================================

def load_pixel_format_from_txt(target_dir):
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_format = "BayerRG12"
    
    if not os.path.exists(settings_path):
        print(f"⚠️ 'camera_all_settings.txt'가 없습니다. 기본값({default_format})으로 진행합니다.")
        return default_format

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            pixel_format = summary.get("PixelFormat", default_format)
            
            print(f"📄 [설정 파일 로드] 픽셀 포맷: {pixel_format}")
            return pixel_format
            
    except Exception as e:
        print(f"⚠️ 설정 파싱 실패 ({e}). 기본값으로 진행합니다.")
        
    return default_format

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

def analyze_ppg():
    sensor_csv = os.path.join(TARGET_DIR, "ppg_sensor.csv")
    camera_csv = os.path.join(TARGET_DIR, "camera_timestamps.csv")
    frames_dir = os.path.join(TARGET_DIR, "frames")
    
    if not os.path.exists(sensor_csv) or not os.path.exists(camera_csv):
        print("❌ CSV 파일을 찾을 수 없습니다. 경로를 확인하세요.")
        return

    pixel_format = load_pixel_format_from_txt(TARGET_DIR)

    print("📂 데이터 및 타임스탬프 로드 중...")
    df_sensor = pd.read_csv(sensor_csv)
    df_cam = pd.read_csv(camera_csv)

    # 1. 공통 시간축(t0) 설정
    t0 = min(df_sensor['Timestamp'].iloc[0], df_cam['Timestamp'].iloc[0])

    # 2. 지정된 시간 구간 마스크 생성
    end_sec = START_SEC + DURATION_SEC
    
    sensor_mask = (df_sensor['Timestamp'] - t0 >= START_SEC) & (df_sensor['Timestamp'] - t0 <= end_sec)
    cam_mask = (df_cam['Timestamp'] - t0 >= START_SEC) & (df_cam['Timestamp'] - t0 <= end_sec)

    df_sensor_cut = df_sensor[sensor_mask]
    df_cam_cut = df_cam[cam_mask]

    if len(df_sensor_cut) == 0 or len(df_cam_cut) == 0:
        print("⚠️ 해당 구간에 데이터가 부족합니다.")
        return

    # =========================================================
    # [1단계] 카메라 영상 ROI 설정 
    # =========================================================
    print(f"\n🎥 [1/2] ROI 설정을 위해 첫 번째 프레임을 불러옵니다...")
    first_frame_idx = int(df_cam_cut['Frame_Index'].iloc[0])
    first_frame_path = os.path.join(frames_dir, f"frame_{first_frame_idx:04d}.tiff")
    
    raw_img = tifffile.imread(first_frame_path)
    
    if pixel_format == "BGR8":
        view_img = raw_img
    elif pixel_format == "BayerRG8":
        view_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
    else: 
        rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
        view_img = (rgb_img >> 4).astype(np.uint8)

    cv2.namedWindow("Select ROI (Drag & Press SPACE)", cv2.WINDOW_NORMAL)
    roi = cv2.selectROI("Select ROI (Drag & Press SPACE)", view_img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()

    x, y, w, h = [int(v) for v in roi]
    
    if w == 0 or h == 0:
        print("⚠️ ROI가 선택되지 않았습니다. 종료합니다.")
        return
        
    x, y = x & ~1, y & ~1 
    print(f"✅ 선택된 ROI: x={x}, y={y}, w={w}, h={h}")

    # =========================================================
    # [2단계] 카메라 타겟 채널 및 센서 데이터 추출
    # =========================================================
    # 💡 사용자가 지정한 채널 인덱스 확인
    channel_idx_map = {'B': 0, 'G': 1, 'R': 2}
    ch_idx = channel_idx_map.get(TARGET_CHANNEL.upper(), 1) # 기본값 G
    
    print(f"⏳ {len(df_cam_cut)}개 프레임에서 '{TARGET_CHANNEL}' 채널 값을 추출 중입니다...")
    cam_values = []
    t_cam = []
    
    for idx, row in df_cam_cut.iterrows():
        f_idx = int(row['Frame_Index'])
        f_time = row['Timestamp'] - t0
        
        f_path = os.path.join(frames_dir, f"frame_{f_idx:04d}.tiff")
        img = tifffile.imread(f_path)
        
        crop_img = img[y:y+h, x:x+w]
        
        if pixel_format == "BGR8":
            bgr_crop = crop_img
        else: 
            bgr_crop = cv2.cvtColor(crop_img, cv2.COLOR_BayerBG2BGR)
        
        # 💡 설정된 채널의 공간 평균 계산
        ch_mean = np.mean(bgr_crop[:, :, ch_idx])
        cam_values.append(ch_mean)
        t_cam.append(f_time)

    t_cam = np.array(t_cam)
    y_cam = np.array(cam_values)

    print(f"📡 센서 데이터를 처리합니다...")
    t_sensor = df_sensor_cut['Timestamp'].values - t0
    y_sensor = df_sensor_cut['IR_Value_Raw'].values if 'IR_Value_Raw' in df_sensor_cut.columns else df_sensor_cut['IR_Value'].values

    # =========================================================
    # [3단계] 신호 필터링 및 피크 검출
    # =========================================================
    fs_cam = 1.0 / np.mean(np.diff(t_cam))
    fs_sensor = 1.0 / np.mean(np.diff(t_sensor))
    
    print(f"⚡ 실제 샘플링: 센서 {fs_sensor:.1f}Hz / 카메라 {fs_cam:.1f}Hz")

    if TARGET_TYPE == 'mouse':
        low_cut, high_cut, min_dist = 2.0, 10.0, 0.05
    else:
        low_cut, high_cut, min_dist = 0.5, 5.0, 0.35

    # 필터링 전용으로 평균을 뺀(Zero-mean) 신호 생성
    y_sensor_centered = y_sensor - np.mean(y_sensor)
    y_cam_centered = y_cam - np.mean(y_cam)

    if INVERT_SENSOR: y_sensor_centered = -y_sensor_centered
    if INVERT_CAMERA: y_cam_centered = -y_cam_centered

    # 필터 적용
    y_sensor_filt = butter_bandpass_filter(y_sensor_centered, low_cut, high_cut, fs_sensor)
    y_cam_filt = butter_bandpass_filter(y_cam_centered, low_cut, high_cut, fs_cam)

    y_sensor_target = y_sensor_filt if SHOW_FILTERED else y_sensor_centered
    y_cam_target = y_cam_filt if SHOW_FILTERED else y_cam_centered

    # 피크 검출
    peaks_sen, _ = find_peaks(y_sensor_target, distance=int(min_dist * fs_sensor), prominence=np.std(y_sensor_target)*0.5)
    peaks_cam, _ = find_peaks(y_cam_target, distance=int(min_dist * fs_cam), prominence=np.std(y_cam_target)*0.5)

    bpm_sen = 60.0 / np.mean(np.diff(t_sensor[peaks_sen])) if len(peaks_sen) > 1 else 0
    bpm_cam = 60.0 / np.mean(np.diff(t_cam[peaks_cam])) if len(peaks_cam) > 1 else 0

    print(f"❤️ 센서 BPM: {bpm_sen:.1f} | 📷 카메라 '{TARGET_CHANNEL}' 채널 BPM: {bpm_cam:.1f}")

    # =========================================================
    # [시각화] 2x2 Grid로 그래프 그리기
    # =========================================================
    fig, axs = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
    ax_raw_sen = axs[0, 0]   
    ax_raw_cam = axs[0, 1]   
    ax_filt_sen = axs[1, 0]  
    ax_filt_cam = axs[1, 1]  
    
    # 💡 동적 채널 이름 및 테마 색상 설정
    ch_name = {'R': 'Red', 'G': 'Green', 'B': 'Blue'}.get(TARGET_CHANNEL.upper(), 'Green')
    ch_color = {'R': '#ef4444', 'G': '#22c55e', 'B': '#3b82f6'}.get(TARGET_CHANNEL.upper(), '#22c55e')

    # --- [상단-좌측] 센서 Raw 신호 ---
    ax_raw_sen.plot(t_sensor, y_sensor, color='gray', linewidth=1.5, label='Absolute Raw Sensor')
    ax_raw_sen.set_title("Hardware IR Sensor (Raw Absolute Value)", fontsize=13, fontweight='bold')
    ax_raw_sen.set_ylabel("Sensor Value")
    ax_raw_sen.legend(loc='upper right')
    ax_raw_sen.grid(True, linestyle='--', alpha=0.6)

    # --- [상단-우측] 카메라 Raw 신호 ---
    ax_raw_cam.plot(t_cam, y_cam, color='gray', linewidth=1.5, label=f'Absolute Raw Camera ({TARGET_CHANNEL}-Ch)')
    ax_raw_cam.set_title(f"Camera rPPG {ch_name} Channel (Raw Pixel Value)", fontsize=13, fontweight='bold')
    ax_raw_cam.set_ylabel("Pixel Intensity")
    ax_raw_cam.legend(loc='upper right')
    ax_raw_cam.grid(True, linestyle='--', alpha=0.6)

    # --- [하단-좌측] 센서 필터링 신호 ---
    ax_filt_sen.plot(t_sensor, y_sensor_target, color='#f43f5e', linewidth=2, label='Filtered Sensor')
    if len(peaks_sen) > 0:
        ax_filt_sen.plot(t_sensor[peaks_sen], y_sensor_target[peaks_sen], "x", color='black', markersize=8, markeredgewidth=2)
    ax_filt_sen.set_title(f"Filtered Sensor Signal | BPM: {bpm_sen:.1f}", fontsize=13, fontweight='bold', color='#f43f5e')
    ax_filt_sen.set_xlabel("Time (seconds)", fontsize=12)
    ax_filt_sen.set_ylabel("Amplitude (Zero-Mean)")
    ax_filt_sen.legend(loc='upper right')
    ax_filt_sen.grid(True, linestyle='--', alpha=0.6)

    # --- [하단-우측] 카메라 필터링 신호 ---
    ax_filt_cam.plot(t_cam, y_cam_target, color=ch_color, linewidth=2, label=f'Filtered Camera ({TARGET_CHANNEL}-Ch)')
    if len(peaks_cam) > 0:
        ax_filt_cam.plot(t_cam[peaks_cam], y_cam_target[peaks_cam], "x", color='black', markersize=8, markeredgewidth=2)
    ax_filt_cam.set_title(f"Filtered Camera Signal | BPM: {bpm_cam:.1f}", fontsize=13, fontweight='bold', color=ch_color)
    ax_filt_cam.set_xlabel("Time (seconds)", fontsize=12)
    ax_filt_cam.set_ylabel("Amplitude (Zero-Mean)")
    ax_filt_cam.legend(loc='upper right')
    ax_filt_cam.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_ppg()

프레임 ONLY , 채널 선택, 영상 지정영역 평균값 동기화, 영상 재생, 저장

In [7]:
# 파일명: analyze_single_channel_sync_video_dual_graph.py
# 지시사항: 지정된 시간 동안의 채널(R/G/B) 픽셀 변화량(Raw)과 필터링된 신호를 추출하고, 원본 영상과 동기화된 두 개의 실시간 그래프를 하단에 결합하여 재생 및 MP4 영상으로 저장합니다.

import os
import cv2
import tifffile
import numpy as np
from scipy.signal import butter, filtfilt
import json

# =========================================================
# [설정] 분석할 폴더 및 옵션
# =========================================================
TARGET_DIR = r"data\basler_0409\20260403_153112"
TARGET_DIR = r"data\basler_0409\20260403_153547"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_153712"    #발바닥
# TARGET_DIR = r"data\basler_0409\20260403_154017"    #발바닥, 카메라 흔들림
TARGET_DIR = r"data\basler_0409\20260403_154225"    #발바닥 *
# TARGET_DIR = r"data\basler_0409\20260403_154512"
# TARGET_DIR = r"data\basler_0409\20260403_154645"    # 배경에 신호
# TARGET_DIR = r"data\basler_0409\20260403_155010"    # 근접
# TARGET_DIR = r"data\basler_0409\20260403_155121"
# TARGET_DIR = r"data\basler_0409\20260403_155212"
# TARGET_DIR = r"data\basler_0409\20260403_155430"    # 근접, 꼬리 , 발바닥
# TARGET_DIR = r"data\basler_0409\20260403_155624"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_155725"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_155936"    #블러
# TARGET_DIR = r"data\basler_0409\20260403_161750"    #비마취
# TARGET_DIR = r"data\basler_0409\20260403_161958"
# TARGET_DIR = r"data\basler_0409\20260403_162203"
# TARGET_DIR = r"data\basler_0409\20260403_162415"
# TARGET_DIR = r"data\basler_0409\20260403_162607"


FRAMES_SUBDIR = "frames" 

# 분석할 단일 채널 선택 ('R', 'G', 'B' 중 택 1)
TARGET_CHANNEL = 'R'  

# 분석 시간 구간 설정
START_SEC = 0       # 분석 시작 시간 (초)
DURATION_SEC = 20     # 분석할 길이 (초)

# 💡 밴드 패스 필터 범위 설정 (BPM 단위)
FILTER_BPM_MIN = 120   # 필터 하한 (BPM)
FILTER_BPM_MAX = 200   # 필터 상한 (BPM)

# 그래프 설정
INVERT_CAMERA = False  # 카메라 신호 뒤집기
GRAPH_HEIGHT = 150     # 하나의 그래프가 차지할 세로 높이 (총 2개가 표시되므로 150 * 2 = 300픽셀 추가됨)
# =========================================================

def load_camera_settings(target_dir):
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_format = "BayerRG12"
    default_fps = 60.0
    
    if not os.path.exists(settings_path):
        print(f"⚠️ 'camera_all_settings.txt'가 없습니다. 기본값(Format: {default_format}, FPS: {default_fps})으로 진행합니다.")
        return default_format, default_fps

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            
            pixel_format = summary.get("PixelFormat", default_format)
            fps = summary.get("FPS_Result", summary.get("FPS_Target", default_fps))
            
            print(f"📄 [설정 파일 로드] 픽셀 포맷: {pixel_format} | 🎥 카메라 FPS: {fps:.2f}")
            return pixel_format, float(fps)
            
    except Exception as e:
        print(f"⚠️ 설정 파싱 실패 ({e}). 기본값으로 진행합니다.")
        
    return default_format, default_fps

def convert_to_8bit_bgr(img, pixel_format):
    """표시 및 영상 저장을 위해 다양한 포맷을 8-bit BGR로 변환하는 헬퍼 함수"""
    if len(img.shape) == 3 and img.shape[2] == 3:
        return img.astype(np.uint8) if img.dtype != np.uint8 else img
    elif pixel_format == "BGR8":
        return img
    elif pixel_format == "BayerRG8":
        return cv2.cvtColor(img, cv2.COLOR_BayerBG2BGR)
    else: 
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BayerBG2BGR)
        return (rgb_img >> 4).astype(np.uint8)

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

def analyze_and_create_sync_video():
    search_dir = os.path.join(TARGET_DIR, FRAMES_SUBDIR)
    if not os.path.exists(search_dir):
        search_dir = TARGET_DIR

    frame_files = sorted([f for f in os.listdir(search_dir) if f.lower().endswith(('.tiff', '.tif', '.png', '.jpg'))])
    
    if not frame_files:
        print(f"❌ '{search_dir}' 경로에서 이미지 파일을 찾을 수 없습니다.")
        return

    pixel_format, camera_fps = load_camera_settings(TARGET_DIR)

    # =========================================================
    # [시간 구간 분할]
    # =========================================================
    total_frames = len(frame_files)
    start_idx = int(START_SEC * camera_fps)
    end_idx = int((START_SEC + DURATION_SEC) * camera_fps)
    
    if start_idx >= total_frames:
        print(f"❌ 시작 시간이 영상 길이를 벗어났습니다.")
        return
        
    end_idx = min(end_idx, total_frames)
    target_frame_files = frame_files[start_idx:end_idx]
    total_target_frames = len(target_frame_files)
    
    print(f"✂️ [구간 설정] {START_SEC}초 ~ {START_SEC+DURATION_SEC}초 (총 {total_target_frames} 프레임)")

    # =========================================================
    # [1단계] ROI 설정
    # =========================================================
    first_frame_path = os.path.join(search_dir, target_frame_files[0])
    
    if first_frame_path.lower().endswith(('.tiff', '.tif')):
        raw_img = tifffile.imread(first_frame_path)
    else:
        raw_img = cv2.imread(first_frame_path, cv2.IMREAD_UNCHANGED)

    view_img = convert_to_8bit_bgr(raw_img, pixel_format)

    cv2.namedWindow("Select ROI (Drag & Press SPACE)", cv2.WINDOW_NORMAL)
    roi = cv2.selectROI("Select ROI (Drag & Press SPACE)", view_img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()

    x, y, w, h = [int(v) for v in roi]
    
    if w == 0 or h == 0:
        print("⚠️ ROI가 선택되지 않았습니다. 종료합니다.")
        return
        
    x, y = x & ~1, y & ~1

    # =========================================================
    # [2단계] 데이터 추출 및 밴드 패스 필터 적용
    # =========================================================
    print(f"⏳ [1/2] 전체 프레임에서 '{TARGET_CHANNEL}' 채널 값을 추출하고 필터링합니다...")
    channel_idx_map = {'B': 0, 'G': 1, 'R': 2}
    ch_idx = channel_idx_map.get(TARGET_CHANNEL.upper(), 1)
    
    raw_vals = []
    
    for f_name in target_frame_files:
        f_path = os.path.join(search_dir, f_name)
        img = tifffile.imread(f_path) if f_path.lower().endswith(('.tiff', '.tif')) else cv2.imread(f_path, cv2.IMREAD_UNCHANGED)
        
        crop_img = img[y:y+h, x:x+w]
        
        if len(crop_img.shape) == 3 and crop_img.shape[2] == 3:
            bgr_crop = crop_img
        elif pixel_format == "BGR8":
            bgr_crop = crop_img
        else:
            bgr_crop = cv2.cvtColor(crop_img, cv2.COLOR_BayerBG2BGR)
            
        val = np.mean(bgr_crop[:, :, ch_idx])
        raw_vals.append(val)

    y_raw = np.array(raw_vals)
    t_cam = np.arange(start_idx, start_idx + total_target_frames) / camera_fps

    # 필터 연산 적용
    low_cut = FILTER_BPM_MIN / 60.0
    high_cut = FILTER_BPM_MAX / 60.0
    
    y_raw_centered = y_raw - np.mean(y_raw)
    if INVERT_CAMERA:
        y_raw_centered = -y_raw_centered
        
    y_filt = butter_bandpass_filter(y_raw_centered, low_cut, high_cut, camera_fps)

    # 스케일링을 위한 최소/최대값 파악 (Raw Data)
    min_raw, max_raw = np.min(y_raw), np.max(y_raw)
    range_raw = max_raw - min_raw if max_raw > min_raw else 1
    min_raw -= range_raw * 0.1
    max_raw += range_raw * 0.1

    # 스케일링을 위한 최소/최대값 파악 (Filtered Data)
    min_filt, max_filt = np.min(y_filt), np.max(y_filt)
    range_filt = max_filt - min_filt if max_filt > min_filt else 1
    min_filt -= range_filt * 0.1
    max_filt += range_filt * 0.1

    # =========================================================
    # [3단계] 동기화된 영상 생성 및 재생
    # =========================================================
    print(f"🎥 [2/2] 동기화 영상을 생성하고 재생합니다...")
    
    # 비디오 저장 설정 (포맷: MP4)
    frame_width = view_img.shape[1]
    frame_height = view_img.shape[0]
    # 그래프 2개가 들어가므로 GRAPH_HEIGHT * 2 공간 확보
    out_height = frame_height + (GRAPH_HEIGHT * 2) 
    
    out_video_path = os.path.join(TARGET_DIR, f"sync_output_{TARGET_CHANNEL}ch_dual_graph.mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(out_video_path, fourcc, camera_fps, (frame_width, out_height))

    # Raw 그래프 선 좌표 계산
    graph_pts_raw = []
    for i, val in enumerate(y_raw):
        pt_x = int(i * frame_width / total_target_frames)
        pt_y = int(GRAPH_HEIGHT - ((val - min_raw) / (max_raw - min_raw)) * GRAPH_HEIGHT)
        graph_pts_raw.append((pt_x, pt_y))
    graph_pts_raw = np.array(graph_pts_raw, dtype=np.int32)

    # Filtered 그래프 선 좌표 계산
    graph_pts_filt = []
    for i, val in enumerate(y_filt):
        pt_x = int(i * frame_width / total_target_frames)
        pt_y = int(GRAPH_HEIGHT - ((val - min_filt) / (max_filt - min_filt)) * GRAPH_HEIGHT)
        graph_pts_filt.append((pt_x, pt_y))
    graph_pts_filt = np.array(graph_pts_filt, dtype=np.int32)

    # 그래프 색상 지정 (Raw: 회색, Filtered: 채널 색상)
    color_map_bgr = {'R': (0, 0, 255), 'G': (0, 255, 0), 'B': (255, 0, 0)}
    color_filt = color_map_bgr.get(TARGET_CHANNEL.upper(), (0, 255, 0))
    color_raw = (180, 180, 180) # 연한 회색

    # 화면에 띄울 창 설정
    window_name = "Sync Player (Press 'q' or 'ESC' to stop)"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    
    delay = int(1000 / camera_fps) if camera_fps > 0 else 16 

    for i, f_name in enumerate(target_frame_files):
        f_path = os.path.join(search_dir, f_name)
        img = tifffile.imread(f_path) if f_path.lower().endswith(('.tiff', '.tif')) else cv2.imread(f_path, cv2.IMREAD_UNCHANGED)
        
        # 1. 상단 원본 프레임 생성
        frame_bgr = convert_to_8bit_bgr(img, pixel_format)
        cv2.rectangle(frame_bgr, (x, y), (x+w, y+h), (0, 255, 255), 2)
        
        # 2. 첫 번째 하단 그래프 영역 (Raw Data)
        img_graph_raw = np.zeros((GRAPH_HEIGHT, frame_width, 3), dtype=np.uint8)
        cv2.polylines(img_graph_raw, [graph_pts_raw], isClosed=False, color=color_raw, thickness=2)
        
        # 3. 두 번째 하단 그래프 영역 (Filtered Data)
        img_graph_filt = np.zeros((GRAPH_HEIGHT, frame_width, 3), dtype=np.uint8)
        cv2.polylines(img_graph_filt, [graph_pts_filt], isClosed=False, color=color_filt, thickness=2)
        
        # 현재 타이밍 커서 (수직 빨간 선)
        curr_x = int(i * frame_width / total_target_frames)
        cv2.line(img_graph_raw, (curr_x, 0), (curr_x, GRAPH_HEIGHT), (0, 0, 255), 2)
        cv2.line(img_graph_filt, (curr_x, 0), (curr_x, GRAPH_HEIGHT), (0, 0, 255), 2)
        
        # 텍스트 표기
        curr_time = t_cam[i]
        cv2.putText(img_graph_raw, f"T: {curr_time:.3f}s | Absolute Raw: {y_raw[i]:.1f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(img_graph_filt, f"T: {curr_time:.3f}s | Filtered AC: {y_filt[i]:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        
        # 두 그래프 사이 경계선 (구분선)
        cv2.line(img_graph_raw, (0, GRAPH_HEIGHT-1), (frame_width, GRAPH_HEIGHT-1), (100, 100, 100), 1)

        # 4. 위(영상) + 중간(Raw 그래프) + 아래(Filtered 그래프) 결합
        composite_frame = np.vstack((frame_bgr, img_graph_raw, img_graph_filt))
        
        # 비디오 저장 및 재생
        video_writer.write(composite_frame)
        cv2.imshow(window_name, composite_frame)
        
        # 프레임레이트에 맞춰 재생
        key = cv2.waitKey(delay) & 0xFF
        if key == 27 or key == ord('q'):
            print("⏹️ 재생이 중단되었습니다.")
            break

    video_writer.release()
    cv2.destroyAllWindows()
    print(f"✅ 동기화 영상이 성공적으로 저장되었습니다: {out_video_path}")

if __name__ == "__main__":
    analyze_and_create_sync_video()

📄 [설정 파일 로드] 픽셀 포맷: BayerRG12 | 🎥 카메라 FPS: 60.00
✂️ [구간 설정] 0초 ~ 20초 (총 1200 프레임)
⏳ [1/2] 전체 프레임에서 'R' 채널 값을 추출하고 필터링합니다...
🎥 [2/2] 동기화 영상을 생성하고 재생합니다...
✅ 동기화 영상이 성공적으로 저장되었습니다: data\basler_0409\20260403_154225\sync_output_Rch_dual_graph.mp4


ROI 2개 지정, raw값 비교

In [ ]:
# 파일명: analyze_dual_roi_raw_separated.py
# 지시사항: 두 개의 ROI를 설정하여 단일 채널(R/G/B) 픽셀 값을 추출합니다. 
# 출력은 총 4단으로 구성됩니다 (1: ROI 설정 이미지, 2: ROI 1 절대 Raw 값, 3: ROI 2 절대 Raw 값, 4: 필터링된 두 신호 겹쳐서 비교).

import os
import cv2
import tifffile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
import json

# =========================================================
# [설정] 분석할 폴더 및 옵션
# =========================================================
# TARGET_DIR = r"data\basler_0409\20260403_153112"
# TARGET_DIR = r"data\basler_0409\20260403_153547"    #발바닥
# TARGET_DIR = r"data\basler_0409\20260403_153712"    #발바닥
# TARGET_DIR = r"data\basler_0409\20260403_154017"    #발바닥, 카메라 흔들림
# TARGET_DIR = r"data\basler_0409\20260403_154225"    #발바닥, 마우스 전체적으로 움직이는 신호(하단 부근 더 뚜렷함)
# TARGET_DIR = r"data\basler_0409\20260403_154512"    #발바닥
# TARGET_DIR = r"data\basler_0409\20260403_154645"    #카메라 흔들림
# TARGET_DIR = r"data\basler_0409\20260403_155010"    #근접
TARGET_DIR = r"data\basler_0409\20260403_155121"
# TARGET_DIR = r"data\basler_0409\20260403_155212"
# TARGET_DIR = r"data\basler_0409\20260403_155430"
# TARGET_DIR = r"data\basler_0409\20260403_155624"  #블러, 배경신호
# TARGET_DIR = r"data\basler_0409\20260403_155725"  #블러, 배경신호
# TARGET_DIR = r"data\basler_0409\20260403_155936"  #블러
# TARGET_DIR = r"data\basler_0409\20260403_161750"  #비마취
# TARGET_DIR = r"data\basler_0409\20260403_161958"
# TARGET_DIR = r"data\basler_0409\20260403_162203"
# TARGET_DIR = r"data\basler_0409\20260403_162415"
# TARGET_DIR = r"data\basler_0409\20260403_162607"

FRAMES_SUBDIR = "frames" 

# 분석할 단일 채널 선택 ('R', 'G', 'B' 중 택 1)
TARGET_CHANNEL = 'R'  

# 분석 시간 구간 설정
START_SEC = 5       # 분석 시작 시간 (초)
DURATION_SEC = 2   # 분석할 길이 (초)

# 💡 밴드 패스 필터 범위 설정 (BPM 단위)
FILTER_BPM_MIN = 100   # 필터 하한 (BPM)
FILTER_BPM_MAX = 200   # 필터 상한 (BPM)

# 그래프 설정
INVERT_CAMERA = False  # 카메라 신호 뒤집기
# =========================================================

def load_camera_settings(target_dir):
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_format = "BayerRG12"
    default_fps = 60.0
    
    if not os.path.exists(settings_path):
        return default_format, default_fps

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            
            pixel_format = summary.get("PixelFormat", default_format)
            fps = summary.get("FPS_Result", summary.get("FPS_Target", default_fps))
            return pixel_format, float(fps)
            
    except Exception:
        pass
        
    return default_format, default_fps

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

def analyze_dual_roi_raw_separated():
    search_dir = os.path.join(TARGET_DIR, FRAMES_SUBDIR)
    if not os.path.exists(search_dir):
        search_dir = TARGET_DIR

    pixel_format, camera_fps = load_camera_settings(TARGET_DIR)

    # =========================================================
    # [시간 구간 분할 및 타임스탬프 로드]
    # =========================================================
    cam_csv_path = os.path.join(TARGET_DIR, "camera_timestamps.csv")
    target_frame_files = []
    t_cam_list = []
    
    if os.path.exists(cam_csv_path):
        print("🕒 'camera_timestamps.csv'에서 실제 하드웨어 시간을 불러옵니다.")
        df_cam = pd.read_csv(cam_csv_path)
        t0 = df_cam['Timestamp'].iloc[0]
        
        end_sec = START_SEC + DURATION_SEC
        mask = (df_cam['Timestamp'] - t0 >= START_SEC) & (df_cam['Timestamp'] - t0 <= end_sec)
        df_cam_cut = df_cam[mask]
        
        for _, row in df_cam_cut.iterrows():
            f_idx = int(row['Frame_Index'])
            f_name = f"frame_{f_idx:04d}.tiff"
            
            if os.path.exists(os.path.join(search_dir, f_name)):
                target_frame_files.append(f_name)
                t_cam_list.append(row['Timestamp'] - t0)
                
        t_cam = np.array(t_cam_list)
        if len(t_cam) > 1:
            camera_fps = 1.0 / np.mean(np.diff(t_cam))
            
    else:
        print(f"⚠️ CSV 파일이 없어 설정 파일의 FPS로 가상 시간을 계산합니다.")
        frame_files = sorted([f for f in os.listdir(search_dir) if f.lower().endswith(('.tiff', '.tif', '.png', '.jpg'))])
        
        total_frames = len(frame_files)
        start_idx = int(START_SEC * camera_fps)
        end_idx = int((START_SEC + DURATION_SEC) * camera_fps)
        
        if start_idx >= total_frames:
            print("❌ 시작 시간이 영상 길이를 벗어났습니다.")
            return
            
        end_idx = min(end_idx, total_frames)
        target_frame_files = frame_files[start_idx:end_idx]
        t_cam = np.arange(start_idx, start_idx + len(target_frame_files)) / camera_fps

    if not target_frame_files:
        print(f"❌ '{search_dir}' 경로에서 분석할 이미지 파일을 찾을 수 없습니다.")
        return

    # =========================================================
    # [1단계] ROI 두 개 설정
    # =========================================================
    first_frame_path = os.path.join(search_dir, target_frame_files[0])
    img_raw = tifffile.imread(first_frame_path) if first_frame_path.lower().endswith(('.tiff', '.tif')) else cv2.imread(first_frame_path, cv2.IMREAD_UNCHANGED)

    if len(img_raw.shape) == 3 and img_raw.shape[2] == 3:
        view_img = img_raw.astype(np.uint8) if img_raw.dtype != np.uint8 else img_raw
    elif pixel_format == "BGR8":
        view_img = img_raw
    elif pixel_format == "BayerRG8":
        view_img = cv2.cvtColor(img_raw, cv2.COLOR_BayerBG2BGR)
    else: 
        view_img = (cv2.cvtColor(img_raw, cv2.COLOR_BayerBG2BGR) >> 4).astype(np.uint8)

    # 첫 번째 ROI 선택 (빨간색 컨셉)
    cv2.namedWindow("Select ROI 1 (Drag & SPACE)", cv2.WINDOW_NORMAL)
    roi1 = cv2.selectROI("Select ROI 1 (Drag & SPACE)", view_img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    cv2.waitKey(1) 

    # 두 번째 ROI 선택 (파란색 컨셉)
    cv2.namedWindow("Select ROI 2 (Drag & SPACE)", cv2.WINDOW_NORMAL)
    roi2 = cv2.selectROI("Select ROI 2 (Drag & SPACE)", view_img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

    x1, y1, w1, h1 = [int(v) & ~1 for v in roi1]
    x2, y2, w2, h2 = [int(v) & ~1 for v in roi2]

    if w1 == 0 or w2 == 0:
        print("⚠️ ROI가 올바르게 선택되지 않았습니다. 종료합니다.")
        return

    # =========================================================
    # [2단계] 두 ROI에서 데이터 각각 추출
    # =========================================================
    print(f"⏳ [2/3] '{TARGET_CHANNEL}' 채널 값을 두 영역에서 각각 추출 중입니다...")
    ch_idx = {'B': 0, 'G': 1, 'R': 2}.get(TARGET_CHANNEL.upper(), 1)
    
    raw_vals1, raw_vals2 = [], []
    
    for f_name in target_frame_files:
        f_path = os.path.join(search_dir, f_name)
        img = tifffile.imread(f_path) if f_path.lower().endswith(('.tiff', '.tif')) else cv2.imread(f_path, cv2.IMREAD_UNCHANGED)
        
        crop1 = img[y1:y1+h1, x1:x1+w1]
        crop2 = img[y2:y2+h2, x2:x2+w2]
        
        if len(img.shape) == 2: 
            bgr1 = cv2.cvtColor(crop1, cv2.COLOR_BayerBG2BGR)
            bgr2 = cv2.cvtColor(crop2, cv2.COLOR_BayerBG2BGR)
        else:
            bgr1, bgr2 = crop1, crop2

        raw_vals1.append(np.mean(bgr1[:, :, ch_idx]))
        raw_vals2.append(np.mean(bgr2[:, :, ch_idx]))

    # 💡 절대 Raw 값 (8비트/12비트 실제 픽셀 밝기)
    y_raw1 = np.array(raw_vals1)
    y_raw2 = np.array(raw_vals2)

    # =========================================================
    # [3단계] 신호 필터링
    # =========================================================
    print(f"⚡ [3/3] 두 신호를 필터링합니다... (기준 FPS: {camera_fps:.1f}Hz)")
    
    # 💡 BPM을 Hz로 변환하여 밴드 패스 필터 주파수 설정
    low_cut = FILTER_BPM_MIN / 60.0
    high_cut = FILTER_BPM_MAX / 60.0
    min_dist = (60.0 / FILTER_BPM_MAX) * 0.5  # 최대 BPM 기준 간격의 절반

    # 필터 왜곡 방지를 위해 연산 시에만 Zero-mean 적용
    y_raw1_c = -(y_raw1 - np.mean(y_raw1)) if INVERT_CAMERA else (y_raw1 - np.mean(y_raw1))
    y_raw2_c = -(y_raw2 - np.mean(y_raw2)) if INVERT_CAMERA else (y_raw2 - np.mean(y_raw2))

    y_filt1 = butter_bandpass_filter(y_raw1_c, low_cut, high_cut, camera_fps)
    y_filt2 = butter_bandpass_filter(y_raw2_c, low_cut, high_cut, camera_fps)

    peaks1, _ = find_peaks(y_filt1, distance=int(min_dist * camera_fps), prominence=np.std(y_filt1)*0.5)
    peaks2, _ = find_peaks(y_filt2, distance=int(min_dist * camera_fps), prominence=np.std(y_filt2)*0.5)

    bpm1 = 60.0 / np.mean(np.diff(t_cam[peaks1])) if len(peaks1) > 1 else 0
    bpm2 = 60.0 / np.mean(np.diff(t_cam[peaks2])) if len(peaks2) > 1 else 0

    # =========================================================
    # [시각화] Jupyter Notebook 4단 분리 그래프 출력
    # =========================================================
    fig = plt.figure(figsize=(14, 18)) # 세로 길이를 살짝 늘림
    
    # --- 1. ROI가 표시된 이미지 ---
    ax_img = plt.subplot(4, 1, 1)
    view_img_rgb = cv2.cvtColor(view_img, cv2.COLOR_BGR2RGB)
    
    cv2.rectangle(view_img_rgb, (x1, y1), (x1+w1, y1+h1), (255, 0, 0), 3) # ROI 1: Red
    cv2.putText(view_img_rgb, "ROI 1", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
    
    cv2.rectangle(view_img_rgb, (x2, y2), (x2+w2, y2+h2), (0, 0, 255), 3) # ROI 2: Blue
    cv2.putText(view_img_rgb, "ROI 2", (x2, y2-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    
    ax_img.imshow(view_img_rgb)
    ax_img.axis('off')
    ax_img.set_title("Selected ROIs (ROI 1: Red, ROI 2: Blue)", fontsize=14, fontweight='bold')

    # --- 2. ROI 1 절대 Raw 데이터 ---
    ax_raw1 = plt.subplot(4, 1, 2)
    ax_raw1.plot(t_cam, y_raw1, color='#ef4444', linewidth=1.5, label='ROI 1 Absolute Raw')
    ax_raw1.set_title(f"ROI 1 Absolute Raw Signal [{TARGET_CHANNEL} Channel]", fontsize=14, fontweight='bold', color='#ef4444')
    ax_raw1.set_ylabel("Pixel Intensity")
    ax_raw1.legend(loc='upper right')
    ax_raw1.grid(True, linestyle='--', alpha=0.6)

    # --- 3. ROI 2 절대 Raw 데이터 ---
    ax_raw2 = plt.subplot(4, 1, 3, sharex=ax_raw1) # X축 공유
    ax_raw2.plot(t_cam, y_raw2, color='#3b82f6', linewidth=1.5, label='ROI 2 Absolute Raw')
    ax_raw2.set_title(f"ROI 2 Absolute Raw Signal [{TARGET_CHANNEL} Channel]", fontsize=14, fontweight='bold', color='#3b82f6')
    ax_raw2.set_ylabel("Pixel Intensity")
    ax_raw2.legend(loc='upper right')
    ax_raw2.grid(True, linestyle='--', alpha=0.6)

    # --- 4. 필터링된 AC 신호 비교 (겹쳐 그리기) ---
    ax_filt = plt.subplot(4, 1, 4, sharex=ax_raw1) # X축 공유
    ax_filt.plot(t_cam, y_filt1, color='#ef4444', linewidth=2, label=f'ROI 1 Filtered (BPM: {bpm1:.1f})')
    ax_filt.plot(t_cam, y_filt2, color='#3b82f6', linewidth=2, label=f'ROI 2 Filtered (BPM: {bpm2:.1f})')
    
    if len(peaks1) > 0:
        ax_filt.plot(t_cam[peaks1], y_filt1[peaks1], "x", color='#991b1b', markersize=8, markeredgewidth=2)
    if len(peaks2) > 0:
        ax_filt.plot(t_cam[peaks2], y_filt2[peaks2], "x", color='#1e40af', markersize=8, markeredgewidth=2)
        
    ax_filt.set_title(f"[{TARGET_CHANNEL} Channel] Filtered AC Signals Comparison | Filter: {FILTER_BPM_MIN}~{FILTER_BPM_MAX} BPM", fontsize=14, fontweight='bold')
    ax_filt.set_xlabel("Time (seconds)", fontsize=12)
    ax_filt.set_ylabel("Amplitude (Zero-Mean)")
    ax_filt.legend(loc='upper right')
    ax_filt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_dual_roi_raw_separated()



ROI 3개 + 배경

In [ ]:
# 파일명: analyze_quad_roi_separated.py
# 지시사항: 총 4개의 영역(ROI 1, 2, 3, Background)을 설정하여 단일 채널(R/G/B) 픽셀 평균값을 추출합니다.
# 출력은 총 7단으로 구성됩니다 (1: ROI 설정 이미지, 2~4: ROI 1~3 개별 절대 Raw, 5: Background 개별 절대 Raw, 6: ROI 1~3 필터 비교, 7: Background 필터).

import os
import cv2
import tifffile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
import json

# =========================================================
# [설정] 분석할 폴더 및 옵션
# =========================================================
TARGET_DIR = r"data\basler_0409\20260403_153112"
TARGET_DIR = r"data\basler_0409\20260403_153547"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_153712"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_154017"    #발바닥, 카메라 흔들림
TARGET_DIR = r"data\basler_0409\20260403_154225"    #발바닥, *
# TARGET_DIR = r"data\basler_0409\20260403_154512"    #발바닥
# TARGET_DIR = r"data\basler_0409\20260403_154645"    #배경 신호
# TARGET_DIR = r"data\basler_0409\20260403_155010"    #근접
# TARGET_DIR = r"data\basler_0409\20260403_155121"
# TARGET_DIR = r"data\basler_0409\20260403_155212"
# TARGET_DIR = r"data\basler_0409\20260403_155430"
# TARGET_DIR = r"data\basler_0409\20260403_155624"  #블러, 배경신호
# TARGET_DIR = r"data\basler_0409\20260403_155725"  #블러, 배경신호
# TARGET_DIR = r"data\basler_0409\20260403_155936"  #블러
# TARGET_DIR = r"data\basler_0409\20260403_161750"  #비마취
# TARGET_DIR = r"data\basler_0409\20260403_161958"
# TARGET_DIR = r"data\basler_0409\20260403_162203"
# TARGET_DIR = r"data\basler_0409\20260403_162415"
# TARGET_DIR = r"data\basler_0409\20260403_162607"

FRAMES_SUBDIR = "frames" 

# 분석할 단일 채널 선택 ('R', 'G', 'B' 중 택 1)
TARGET_CHANNEL = 'R'  

# 분석 시간 구간 설정
START_SEC = 5       # 분석 시작 시간 (초)
DURATION_SEC = 10   # 분석할 길이 (초)

# 💡 밴드 패스 필터 범위 설정 (BPM 단위)
FILTER_BPM_MIN = 60   # 필터 하한 (BPM)
FILTER_BPM_MAX = 100   # 필터 상한 (BPM)

# 그래프 설정
INVERT_CAMERA = False  # 카메라 신호 뒤집기
# =========================================================

def load_camera_settings(target_dir):
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_format = "BayerRG12"
    default_fps = 60.0
    
    if not os.path.exists(settings_path):
        return default_format, default_fps

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            
            pixel_format = summary.get("PixelFormat", default_format)
            fps = summary.get("FPS_Result", summary.get("FPS_Target", default_fps))
            return pixel_format, float(fps)
            
    except Exception:
        pass
        
    return default_format, default_fps

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

def analyze_quad_roi_separated():
    search_dir = os.path.join(TARGET_DIR, FRAMES_SUBDIR)
    if not os.path.exists(search_dir):
        search_dir = TARGET_DIR

    pixel_format, camera_fps = load_camera_settings(TARGET_DIR)

    # =========================================================
    # [시간 구간 분할 및 타임스탬프 로드]
    # =========================================================
    cam_csv_path = os.path.join(TARGET_DIR, "camera_timestamps.csv")
    target_frame_files = []
    t_cam_list = []
    
    if os.path.exists(cam_csv_path):
        print("🕒 'camera_timestamps.csv'에서 실제 하드웨어 시간을 불러옵니다.")
        df_cam = pd.read_csv(cam_csv_path)
        t0 = df_cam['Timestamp'].iloc[0]
        
        end_sec = START_SEC + DURATION_SEC
        mask = (df_cam['Timestamp'] - t0 >= START_SEC) & (df_cam['Timestamp'] - t0 <= end_sec)
        df_cam_cut = df_cam[mask]
        
        for _, row in df_cam_cut.iterrows():
            f_idx = int(row['Frame_Index'])
            f_name = f"frame_{f_idx:04d}.tiff"
            
            if os.path.exists(os.path.join(search_dir, f_name)):
                target_frame_files.append(f_name)
                t_cam_list.append(row['Timestamp'] - t0)
                
        t_cam = np.array(t_cam_list)
        if len(t_cam) > 1:
            camera_fps = 1.0 / np.mean(np.diff(t_cam))
            
    else:
        print(f"⚠️ CSV 파일이 없어 설정 파일의 FPS로 가상 시간을 계산합니다.")
        frame_files = sorted([f for f in os.listdir(search_dir) if f.lower().endswith(('.tiff', '.tif', '.png', '.jpg'))])
        
        total_frames = len(frame_files)
        start_idx = int(START_SEC * camera_fps)
        end_idx = int((START_SEC + DURATION_SEC) * camera_fps)
        
        if start_idx >= total_frames:
            print("❌ 시작 시간이 영상 길이를 벗어났습니다.")
            return
            
        end_idx = min(end_idx, total_frames)
        target_frame_files = frame_files[start_idx:end_idx]
        t_cam = np.arange(start_idx, start_idx + len(target_frame_files)) / camera_fps

    if not target_frame_files:
        print(f"❌ '{search_dir}' 경로에서 분석할 이미지 파일을 찾을 수 없습니다.")
        return

    # =========================================================
    # [1단계] 4개의 ROI 설정 (1, 2, 3, Background)
    # =========================================================
    first_frame_path = os.path.join(search_dir, target_frame_files[0])
    img_raw = tifffile.imread(first_frame_path) if first_frame_path.lower().endswith(('.tiff', '.tif')) else cv2.imread(first_frame_path, cv2.IMREAD_UNCHANGED)

    if len(img_raw.shape) == 3 and img_raw.shape[2] == 3:
        view_img = img_raw.astype(np.uint8) if img_raw.dtype != np.uint8 else img_raw
    elif pixel_format == "BGR8":
        view_img = img_raw
    elif pixel_format == "BayerRG8":
        view_img = cv2.cvtColor(img_raw, cv2.COLOR_BayerBG2BGR)
    else: 
        view_img = (cv2.cvtColor(img_raw, cv2.COLOR_BayerBG2BGR) >> 4).astype(np.uint8)

    rois = []
    roi_names = ["ROI 1", "ROI 2", "ROI 3", "Background"]
    
    for name in roi_names:
        cv2.namedWindow(f"Select {name} (Drag & SPACE)", cv2.WINDOW_NORMAL)
        r = cv2.selectROI(f"Select {name} (Drag & SPACE)", view_img, fromCenter=False, showCrosshair=True)
        cv2.destroyAllWindows()
        cv2.waitKey(1) 
        x, y, w, h = [int(v) & ~1 for v in r]
        
        if w == 0 or h == 0:
            print(f"⚠️ {name}가 올바르게 선택되지 않았습니다. 분석을 취소합니다.")
            return
        rois.append((x, y, w, h))

    # =========================================================
    # [2단계] 4개 영역에서 데이터 각각 추출
    # =========================================================
    print(f"⏳ [2/3] '{TARGET_CHANNEL}' 채널 값을 4개 영역에서 추출 중입니다...")
    ch_idx = {'B': 0, 'G': 1, 'R': 2}.get(TARGET_CHANNEL.upper(), 1)
    
    raw_vals = [[] for _ in range(4)]
    
    for f_name in target_frame_files:
        f_path = os.path.join(search_dir, f_name)
        img = tifffile.imread(f_path) if f_path.lower().endswith(('.tiff', '.tif')) else cv2.imread(f_path, cv2.IMREAD_UNCHANGED)
        
        for i, (x, y, w, h) in enumerate(rois):
            crop = img[y:y+h, x:x+w]
            if len(img.shape) == 2: 
                bgr_crop = cv2.cvtColor(crop, cv2.COLOR_BayerBG2BGR)
            else:
                bgr_crop = crop
            raw_vals[i].append(np.mean(bgr_crop[:, :, ch_idx]))

    # 절대 Raw 값 NumPy 배열화
    y_raw = [np.array(vals) for vals in raw_vals]

    # =========================================================
    # [3단계] 신호 필터링
    # =========================================================
    print(f"⚡ [3/3] 4개의 신호를 필터링합니다... (기준 FPS: {camera_fps:.1f}Hz)")
    
    low_cut = FILTER_BPM_MIN / 60.0
    high_cut = FILTER_BPM_MAX / 60.0
    min_dist = (60.0 / FILTER_BPM_MAX) * 0.5

    y_filt = []
    peaks_list = []
    bpms = []

    for i in range(4):
        # Zero-mean 및 Invert 옵션 적용
        y_c = -(y_raw[i] - np.mean(y_raw[i])) if INVERT_CAMERA else (y_raw[i] - np.mean(y_raw[i]))
        
        yf = butter_bandpass_filter(y_c, low_cut, high_cut, camera_fps)
        y_filt.append(yf)
        
        p, _ = find_peaks(yf, distance=int(min_dist * camera_fps), prominence=np.std(yf)*0.5)
        peaks_list.append(p)
        
        bpm = 60.0 / np.mean(np.diff(t_cam[p])) if len(p) > 1 else 0
        bpms.append(bpm)

    # =========================================================
    # [시각화] Jupyter Notebook 7단 분리 그래프 출력
    # =========================================================
    fig = plt.figure(figsize=(14, 28)) # 그래프가 많아졌으므로 세로 길이 증가
    plot_colors = ['#ef4444', '#22c55e', '#3b82f6', '#8b5cf6'] # Red, Green, Blue, Purple(BG)
    box_colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (139, 92, 246)] # RGB for cv2 drawing
    
    # --- 1. ROI가 표시된 이미지 ---
    ax_img = plt.subplot(7, 1, 1)
    view_img_rgb = cv2.cvtColor(view_img, cv2.COLOR_BGR2RGB)
    
    for i, (x, y, w, h) in enumerate(rois):
        cv2.rectangle(view_img_rgb, (x, y), (x+w, y+h), box_colors[i], 3)
        cv2.putText(view_img_rgb, roi_names[i], (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, box_colors[i], 2)
        
    ax_img.imshow(view_img_rgb)
    ax_img.axis('off')
    ax_img.set_title("Selected ROIs (1, 2, 3: Targets, 4: Background)", fontsize=14, fontweight='bold')

    # --- 2. ROI 1 절대 Raw 데이터 ---
    ax_raw_1 = plt.subplot(7, 1, 2)
    ax_raw_1.plot(t_cam, y_raw[0], color=plot_colors[0], linewidth=1.5, label='ROI 1 Raw')
    ax_raw_1.set_title(f"ROI 1 Absolute Raw Signal [{TARGET_CHANNEL} Channel]", fontsize=14, fontweight='bold', color=plot_colors[0])
    ax_raw_1.set_ylabel("Pixel Intensity")
    ax_raw_1.legend(loc='upper right')
    ax_raw_1.grid(True, linestyle='--', alpha=0.6)

    # --- 3. ROI 2 절대 Raw 데이터 ---
    ax_raw_2 = plt.subplot(7, 1, 3, sharex=ax_raw_1)
    ax_raw_2.plot(t_cam, y_raw[1], color=plot_colors[1], linewidth=1.5, label='ROI 2 Raw')
    ax_raw_2.set_title(f"ROI 2 Absolute Raw Signal [{TARGET_CHANNEL} Channel]", fontsize=14, fontweight='bold', color=plot_colors[1])
    ax_raw_2.set_ylabel("Pixel Intensity")
    ax_raw_2.legend(loc='upper right')
    ax_raw_2.grid(True, linestyle='--', alpha=0.6)

    # --- 4. ROI 3 절대 Raw 데이터 ---
    ax_raw_3 = plt.subplot(7, 1, 4, sharex=ax_raw_1)
    ax_raw_3.plot(t_cam, y_raw[2], color=plot_colors[2], linewidth=1.5, label='ROI 3 Raw')
    ax_raw_3.set_title(f"ROI 3 Absolute Raw Signal [{TARGET_CHANNEL} Channel]", fontsize=14, fontweight='bold', color=plot_colors[2])
    ax_raw_3.set_ylabel("Pixel Intensity")
    ax_raw_3.legend(loc='upper right')
    ax_raw_3.grid(True, linestyle='--', alpha=0.6)

    # --- 5. Background 절대 Raw 데이터 ---
    ax_raw_b = plt.subplot(7, 1, 5, sharex=ax_raw_1)
    ax_raw_b.plot(t_cam, y_raw[3], color=plot_colors[3], linewidth=1.5, label='Background Raw')
    ax_raw_b.set_title(f"Background Absolute Raw Signal [{TARGET_CHANNEL} Channel]", fontsize=14, fontweight='bold', color=plot_colors[3])
    ax_raw_b.set_ylabel("Pixel Intensity")
    ax_raw_b.legend(loc='upper right')
    ax_raw_b.grid(True, linestyle='--', alpha=0.6)

    # --- 6. ROI 1~3 필터링된 AC 신호 비교 (겹쳐 그리기) ---
    ax_filt_t = plt.subplot(7, 1, 6, sharex=ax_raw_1)
    for i in range(3):
        ax_filt_t.plot(t_cam, y_filt[i], color=plot_colors[i], linewidth=2, label=f'ROI {i+1} Filtered (BPM: {bpms[i]:.1f})')
        if len(peaks_list[i]) > 0:
            ax_filt_t.plot(t_cam[peaks_list[i]], y_filt[i][peaks_list[i]], "x", color=plot_colors[i], markersize=6, markeredgewidth=2)
            
    ax_filt_t.set_title(f"[{TARGET_CHANNEL} Channel] Targets Filtered Comparison | Filter: {FILTER_BPM_MIN}~{FILTER_BPM_MAX} BPM", fontsize=14, fontweight='bold')
    ax_filt_t.set_ylabel("Amplitude (Zero-Mean)")
    ax_filt_t.legend(loc='upper right')
    ax_filt_t.grid(True, linestyle='--', alpha=0.6)

    # --- 7. Background 필터링된 AC 신호 ---
    ax_filt_b = plt.subplot(7, 1, 7, sharex=ax_raw_1)
    ax_filt_b.plot(t_cam, y_filt[3], color=plot_colors[3], linewidth=2, label=f'Background Filtered (BPM: {bpms[3]:.1f})')
    if len(peaks_list[3]) > 0:
        ax_filt_b.plot(t_cam[peaks_list[3]], y_filt[3][peaks_list[3]], "x", color=plot_colors[3], markersize=6, markeredgewidth=2)
        
    ax_filt_b.set_title(f"[{TARGET_CHANNEL} Channel] Background Filtered Signal", fontsize=14, fontweight='bold', color=plot_colors[3])
    ax_filt_b.set_xlabel("Time (seconds)", fontsize=12)
    ax_filt_b.set_ylabel("Amplitude (Zero-Mean)")
    ax_filt_b.legend(loc='upper right')
    ax_filt_b.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_quad_roi_separated()

픽셀 삭제 - 단일 채널

In [ ]:
# 파일명: analyze_single_channel_absolute_raw.py
# 지시사항: 지정된 시간 구간 및 단일 채널(R/G/B)의 프레임을 분석합니다. 
# 출력은 총 4단으로 구성됩니다. (1: 단순 평균 Raw, 2: 절사평균 Raw, 3: 필터링된 신호, 4: FFT 파워 스펙트럼)

import os
import cv2
import tifffile
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, butter, filtfilt
from scipy.stats import trim_mean  # 절사평균 계산을 위한 모듈 추가
import json

# =========================================================
# [설정] 분석할 폴더 및 옵션
# =========================================================
TARGET_DIR = r"data\basler_0409\20260403_153112"
TARGET_DIR = r"data\basler_0409\20260403_153547"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_153712"    #발바닥
TARGET_DIR = r"data\basler_0409\20260403_154017"    #발바닥, 카메라 흔들림
TARGET_DIR = r"data\basler_0409\20260403_154225"    #발바닥, *
# TARGET_DIR = r"data\basler_0409\20260403_154512"    #발바닥
# TARGET_DIR = r"data\basler_0409\20260403_154645"    #배경 신호
# TARGET_DIR = r"data\basler_0409\20260403_155010"    #근접
# TARGET_DIR = r"data\basler_0409\20260403_155121"
# TARGET_DIR = r"data\basler_0409\20260403_155212"
# TARGET_DIR = r"data\basler_0409\20260403_155430"
# TARGET_DIR = r"data\basler_0409\20260403_155624"  #블러, 배경신호
# TARGET_DIR = r"data\basler_0409\20260403_155725"  #블러, 배경신호
# TARGET_DIR = r"data\basler_0409\20260403_155936"  #블러
# TARGET_DIR = r"data\basler_0409\20260403_161750"  #비마취
# TARGET_DIR = r"data\basler_0409\20260403_161958"
# TARGET_DIR = r"data\basler_0409\20260403_162203"
# TARGET_DIR = r"data\basler_0409\20260403_162415"
# TARGET_DIR = r"data\basler_0409\20260403_162607"


FRAMES_SUBDIR = "frames" 

# 분석할 단일 채널 선택 ('R', 'G', 'B' 중 택 1)
TARGET_CHANNEL = 'R'  

# 분석 시간 구간 설정
START_SEC = 5       # 분석 시작 시간 (초)
DURATION_SEC = 10   # 분석할 길이 (초)

# 💡 밴드 패스 필터 범위 설정 (BPM 단위)
FILTER_BPM_MIN = 60   # 필터 하한 (BPM)
FILTER_BPM_MAX = 150   # 필터 상한 (BPM)

# 💡 절사평균(Trimmed Mean) 비율 설정
TRIM_PERCENT = 20      # 상하위 제거할 픽셀 비율 (%) - 예: 10이면 가장 밝은 10%, 가장 어두운 10% 제외

# 그래프 설정
INVERT_CAMERA = False  # 카메라 신호 뒤집기
# =========================================================

def load_camera_settings(target_dir):
    settings_path = os.path.join(target_dir, "camera_all_settings.txt")
    default_format = "BayerRG12"
    default_fps = 60.0
    
    if not os.path.exists(settings_path):
        print(f"⚠️ 'camera_all_settings.txt'가 없습니다. 기본값(Format: {default_format}, FPS: {default_fps})으로 진행합니다.")
        return default_format, default_fps

    try:
        with open(settings_path, 'r', encoding='utf-8') as f:
            content = f.read()
            
        start_marker = "=== [Summary] Main Camera Settings ==="
        end_marker = "=== [Full Dump] Camera All Settings ==="
        
        start_idx = content.find(start_marker)
        end_idx = content.find(end_marker)
        
        if start_idx != -1 and end_idx != -1:
            json_str = content[start_idx + len(start_marker):end_idx].strip()
            summary = json.loads(json_str)
            
            pixel_format = summary.get("PixelFormat", default_format)
            fps = summary.get("FPS_Result", summary.get("FPS_Target", default_fps))
            
            print(f"📄 [설정 파일 로드] 픽셀 포맷: {pixel_format} | 🎥 카메라 FPS: {fps:.2f}")
            return pixel_format, float(fps)
            
    except Exception as e:
        print(f"⚠️ 설정 파싱 실패 ({e}). 기본값으로 진행합니다.")
        
    return default_format, default_fps

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = filtfilt(b, a, data)
    return y

def analyze_single_channel_from_frames():
    search_dir = os.path.join(TARGET_DIR, FRAMES_SUBDIR)
    if not os.path.exists(search_dir):
        print(f"❌ '{search_dir}' 경로에서 이미지 파일을 찾을 수 없습니다.")
        return

    frame_files = sorted([f for f in os.listdir(search_dir) if f.lower().endswith(('.tiff', '.tif', '.png', '.jpg'))])
    
    if not frame_files:
        print(f"❌ '{search_dir}' 경로에서 이미지 파일을 찾을 수 없습니다.")
        return

    pixel_format, camera_fps = load_camera_settings(TARGET_DIR)

    # =========================================================
    # [시간 구간 분할] 
    # =========================================================
    total_frames = len(frame_files)
    start_idx = int(START_SEC * camera_fps)
    end_idx = int((START_SEC + DURATION_SEC) * camera_fps)
    
    if start_idx >= total_frames:
        print(f"❌ 시작 시간이 영상 길이를 벗어났습니다.")
        return
        
    end_idx = min(end_idx, total_frames)
    target_frame_files = frame_files[start_idx:end_idx]
    
    print(f"✂️ [구간 설정] {START_SEC}초 ~ {START_SEC+DURATION_SEC}초 (프레임: {start_idx} ~ {end_idx-1})")

    # =========================================================
    # [1단계] ROI 설정
    # =========================================================
    first_frame_path = os.path.join(search_dir, target_frame_files[0])
    
    if first_frame_path.lower().endswith(('.tiff', '.tif')):
        raw_img = tifffile.imread(first_frame_path)
    else:
        raw_img = cv2.imread(first_frame_path, cv2.IMREAD_UNCHANGED)

    if len(raw_img.shape) == 3 and raw_img.shape[2] == 3:
        view_img = raw_img
    elif pixel_format == "BGR8":
        view_img = raw_img
    elif pixel_format == "BayerRG8":
        view_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
    else: 
        rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BayerBG2BGR)
        view_img = (rgb_img >> 4).astype(np.uint8)

    cv2.namedWindow("Select ROI (Drag & Press SPACE)", cv2.WINDOW_NORMAL)
    roi = cv2.selectROI("Select ROI (Drag & Press SPACE)", view_img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

    x, y, w, h = [int(v) for v in roi]
    
    if w == 0 or h == 0:
        print("⚠️ ROI가 선택되지 않았습니다. 종료합니다.")
        return
        
    x, y = x & ~1, y & ~1

    # =========================================================
    # [2단계] 타겟 프레임에서 선택한 단일 채널만 추출 (단순 평균 vs 절사평균)
    # =========================================================
    print(f"⏳ [2/3] {len(target_frame_files)}개 프레임에서 '{TARGET_CHANNEL}' 채널 값 추출 중입니다...")
    
    channel_idx_map = {'B': 0, 'G': 1, 'R': 2}
    ch_idx = channel_idx_map.get(TARGET_CHANNEL.upper(), 1)
    
    raw_vals_untrimmed = []
    raw_vals_trimmed = []
    
    for f_name in target_frame_files:
        f_path = os.path.join(search_dir, f_name)
        
        if f_name.lower().endswith(('.tiff', '.tif')):
            img = tifffile.imread(f_path)
        else:
            img = cv2.imread(f_path, cv2.IMREAD_UNCHANGED)
        
        crop_img = img[y:y+h, x:x+w]
        
        if len(crop_img.shape) == 3 and crop_img.shape[2] == 3:
            bgr_crop = crop_img
        elif pixel_format == "BGR8":
            bgr_crop = crop_img
        else:
            bgr_crop = cv2.cvtColor(crop_img, cv2.COLOR_BayerBG2BGR)
        
        # 해당 채널의 픽셀값들을 1차원 배열로 펼침
        pixels = bgr_crop[:, :, ch_idx].flatten()
        
        # 1. 픽셀 선별을 안한 단순 평균 (Untrimmed)
        u_mean = np.mean(pixels)
        raw_vals_untrimmed.append(u_mean)
        
        # 2. 극단값을 제거한 절사평균 (Trimmed)
        t_mean = trim_mean(pixels, TRIM_PERCENT / 100.0)
        raw_vals_trimmed.append(t_mean)

    y_raw_untrimmed = np.array(raw_vals_untrimmed)
    y_raw_trimmed = np.array(raw_vals_trimmed)
    
    # 이후 신호 처리 과정은 절사평균(Trimmed) 데이터를 기준으로 수행
    y_raw = y_raw_trimmed 
    t_cam = np.arange(start_idx, start_idx + len(target_frame_files)) / camera_fps

    # =========================================================
    # [3단계] 신호 필터링 및 피크 검출
    # =========================================================
    print(f"⚡ [3/3] 신호를 필터링하고 주파수를 분석합니다...")

    # BPM을 Hz로 변환하여 밴드 패스 필터 주파수 설정
    low_cut = FILTER_BPM_MIN / 60.0
    high_cut = FILTER_BPM_MAX / 60.0
    min_dist = (60.0 / FILTER_BPM_MAX) * 0.5  # 최대 BPM 기준 간격의 절반

    y_raw_centered = y_raw - np.mean(y_raw)
    if INVERT_CAMERA:
        y_raw_centered = -y_raw_centered

    # 신호 필터링
    y_filt = butter_bandpass_filter(y_raw_centered, low_cut, high_cut, camera_fps)
    peaks, _ = find_peaks(y_filt, distance=int(min_dist * camera_fps), prominence=np.std(y_filt)*0.5)

    bpm_peak = 60.0 / np.mean(np.diff(t_cam[peaks])) if len(peaks) > 1 else 0

    # =========================================================
    # [4단계] FFT(고속 푸리에 변환)를 통한 파워 스펙트럼 계산
    # =========================================================
    n = len(y_filt)
    dt = 1.0 / camera_fps
    freqs = np.fft.fftfreq(n, d=dt)
    fft_vals = np.fft.fft(y_filt)

    power = np.abs(fft_vals) ** 2

    # 양수 주파수 영역만 필터링 (DC 성분 0Hz 제외)
    pos_mask = freqs > 0
    pos_freqs_bpm = freqs[pos_mask] * 60.0
    pos_power = power[pos_mask]

    if len(pos_power) > 0:
        max_idx = np.argmax(pos_power)
        bpm_fft = pos_freqs_bpm[max_idx]
    else:
        bpm_fft = 0

    print(f"❤️ 측정 결과 -> Peak 기반 BPM: {bpm_peak:.1f} | FFT 기반 우세 BPM: {bpm_fft:.1f}")

    # =========================================================
    # [시각화] 4단 구성 그래프 
    # (1: Untrimmed Raw, 2: Trimmed Raw, 3: Filtered Data, 4: FFT)
    # =========================================================
    fig = plt.figure(figsize=(14, 16)) # 세로 길이를 늘려 4개의 그래프 배치
    
    color_map = {'R': '#ef4444', 'G': '#22c55e', 'B': '#3b82f6'}
    ch_color = color_map.get(TARGET_CHANNEL.upper(), '#22c55e')
    ch_name = {'R': 'Red', 'G': 'Green', 'B': 'Blue'}.get(TARGET_CHANNEL.upper(), 'Green')

    # --- [상단 1] 픽셀 선별 안한 Raw 신호 (Simple Mean) ---
    ax_raw_un = plt.subplot(4, 1, 1)
    ax_raw_un.plot(t_cam, y_raw_untrimmed, color='orange', alpha=0.9, linewidth=1.5, label=f'Untrimmed Raw {ch_name}')
    ax_raw_un.set_title(f"[{ch_name} Channel] Absolute Raw Pixel Value (No Pixel Selection)", fontsize=14, fontweight='bold', color='orange')
    ax_raw_un.set_ylabel("Pixel Intensity")
    ax_raw_un.legend(loc='upper right')
    ax_raw_un.grid(True, linestyle='--', alpha=0.6)

    # --- [상단 2] 절사평균 Raw 신호 (Trimmed Mean) ---
    ax_raw_tr = plt.subplot(4, 1, 2, sharex=ax_raw_un)
    ax_raw_tr.plot(t_cam, y_raw_trimmed, color='gray', alpha=0.9, linewidth=1.5, label=f'Trimmed Raw {ch_name} ({TRIM_PERCENT}%)')
    ax_raw_tr.set_title(f"[{ch_name} Channel] Absolute Raw Pixel Value (Trimmed Mean {TRIM_PERCENT}%)", fontsize=14, fontweight='bold')
    ax_raw_tr.set_ylabel("Pixel Intensity")
    ax_raw_tr.legend(loc='upper right')
    ax_raw_tr.grid(True, linestyle='--', alpha=0.6)

    # --- [중간] 필터링 신호 및 피크 그래프 ---
    ax_filt = plt.subplot(4, 1, 3, sharex=ax_raw_un)
    ax_filt.plot(t_cam, y_filt, color=ch_color, linewidth=2, label=f'Filtered {ch_name}')
    if len(peaks) > 0:
        ax_filt.plot(t_cam[peaks], y_filt[peaks], "x", color='red', markersize=8, markeredgewidth=2, label='Peaks')
    ax_filt.set_title(f"[{ch_name} Channel] Filtered AC Signal | Peak BPM: {bpm_peak:.1f}", fontsize=14, fontweight='bold', color=ch_color)
    ax_filt.set_xlabel("Time (seconds)", fontsize=12)
    ax_filt.set_ylabel("Amplitude (Zero-Mean)")
    ax_filt.legend(loc='upper right')
    ax_filt.grid(True, linestyle='--', alpha=0.6)

    # --- [하단] FFT 파워 스펙트럼 그래프 ---
    ax_fft = plt.subplot(4, 1, 4)
    ax_fft.plot(pos_freqs_bpm, pos_power, color='purple', linewidth=1.5, label='FFT Power')
    
    # 설정한 필터 밴드 범위 마킹
    ax_fft.axvline(x=FILTER_BPM_MIN, color='black', linestyle=':', alpha=0.5, label='Filter Min')
    ax_fft.axvline(x=FILTER_BPM_MAX, color='black', linestyle=':', alpha=0.5, label='Filter Max')
    
    # 우세 주파수 마커 표시
    if len(pos_power) > 0:
        ax_fft.plot(bpm_fft, pos_power[max_idx], "o", color='red', markersize=6, label=f'Dominant: {bpm_fft:.1f} BPM')

    ax_fft.set_title(f"[{ch_name} Channel] FFT Power Spectrum | FFT BPM: {bpm_fft:.1f}", fontsize=14, fontweight='bold', color='purple')
    ax_fft.set_xlabel("Frequency (BPM)", fontsize=12)
    ax_fft.set_ylabel("Power")
    
    # X축 범위를 설정한 BPM 기준 주변으로 여유 있게 설정
    pad_bpm = (FILTER_BPM_MAX - FILTER_BPM_MIN) * 0.2
    ax_fft.set_xlim(max(0, FILTER_BPM_MIN - pad_bpm), FILTER_BPM_MAX + pad_bpm)
    ax_fft.legend(loc='upper right')
    ax_fft.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    analyze_single_channel_from_frames()